# 🚀 Step 2 (vLLM): Qwen3.8-27B AWQ + cloudflared 公開APIサーバー

`twolven/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-AWQ-MTP` を **vLLM** で T4 x2 に載せ、cloudflared で公開します。

---

### 前提条件（これだけ）

1. **Accelerator**: `GPU T4 x2`
2. **Internet**: `On`
3. **Add Data**: 以下を追加
   - `yamahei21python/qwen38-27b-uncensored-hauhaucs-awq-mtp`（AWQ 4bit モデル 約19 GB）
   - `my-cloudflared-bin`（cloudflared 40 MB）— 無いと毎回 GitHub から落とす
> ✅ **Secrets の設定は不要です。**（cloudflared はアカウント不要）
>
> 🔓 認証はありません（URL を知れば誰でも使えます）。
> 他人に使われたくないときは環境変数 `API_PASS` を設定してください。
>
> ⚠ 公開URLは **実行のたびに変わります**（quick tunnel の仕様）。
> 現在の URL はビーコン（`python3 -m cli beacon`）から読めます。

---

### Ollama 版との違い

| 項目 | Ollama 版 | このノート（vLLM） |
|---|---|---|
| 量子化 | GGUF Q4_K_M | AWQ W4A16 |
| 推論 | llama.cpp | vLLM（PagedAttention） |
| MTP | なし | `--speculative-config` で有効化（失敗時は自動で無効） |
| 速度 | 13.7 t/s 実測 | T4 x2 では 10〜25 t/s 見込み |

> ℹ T4 は sm75 のため **Marlin カーネルと FP8 KV cache は使えません**。
> AWQ は Triton 実装にフォールバックします。


---
### 🧩 共通ランタイム（proxy / tunnel / keepalive / verify）

`runtime/` の真実源をビルダーが連結して埋め込んでいます。
セルを直接編集しても次のビルドで上書きされます。


In [ ]:
# ============================================================
# 共通ランタイム（runtime/*.py から自動生成。手で編集しない）
# proxy / tunnel / keepalive / verify の配管をここに集約する。
# ============================================================
# ── runtime/net.py ────────────────────────────────────────
"""トンネル URL 取得と HTTP 疎通の共通処理（生成コードへ埋め込む）。"""
import json
import re
import socket
import time
import urllib.error
import urllib.request

# hosts は表駆動。今はログ解析の 1 種類しか無いが、待ち時間を URL の出所と
# 一緒に置いておくと `kc_wait_url` が既定値を知らずに済む。表に無い名前は
# KeyError になる（黙って空 URL を返さない）。
TUNNEL_HOSTS = {
    "cloudflared": {
        "url_re": re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com"),
        "wait_seconds": 90,
    },
}


# Dataset 同梱バイナリと同じ版に固定する（latest は再現性がない）。
KC_CLOUDFLARED_VERSION = "2026.10.4"


def kc_cloudflared_url(version=KC_CLOUDFLARED_VERSION):
    """cloudflared (linux amd64) のダウンロード URL。"""
    return ("https://github.com/cloudflare/cloudflared/releases/download/"
            f"{version}/cloudflared-linux-amd64")


def kc_probe_url(host, log_path=None):
    """現在の公開URLを返す（無ければ ""）。host は cloudflared のみ。"""
    spec = TUNNEL_HOSTS[host]
    if not log_path:
        return ""
    try:
        with open(log_path) as f:
            m = spec["url_re"].search(f.read())
        return m.group(0) if m else ""
    except OSError:
        return ""


def kc_wait_url(host, log_path=None, seconds=None, interval=1, progress_every=10):
    """公開URLが出るまで待つ。seconds 未指定なら host ごとの既定値。"""
    if seconds is None:
        seconds = TUNNEL_HOSTS[host]["wait_seconds"]
    for i in range(seconds):
        url = kc_probe_url(host, log_path)
        if url:
            return url
        if progress_every and i % progress_every == progress_every - 1:
            print(f"⏳ URL 待ち ({i + 1}/{seconds})...")
        time.sleep(interval)
    return ""


def kc_urlopen_v4(url, timeout=15):
    """公開 URL を開く。失敗したら IPv4 優先で名前解決して 1 回だけ再試行。

    Kaggle の VM は AAAA（IPv6）を返すのに IPv6 経路が無いことがあり、
    `urlopen` が "Network is unreachable" で落ちる（dryrun で実測）。
    自分の URL に届かないだけでトンネルは生きているので、ここで諦めると
    keepalive が**生きているトンネルを再起動して URL が変わってしまう**。
    """
    try:
        return urllib.request.urlopen(url, timeout=timeout)
    except Exception as first:
        err = first
    orig = socket.getaddrinfo

    def _v4(host, port, family=0, type=0, proto=0, flags=0):
        return orig(host, port, socket.AF_INET, type, proto, flags)

    socket.getaddrinfo = _v4
    try:
        return urllib.request.urlopen(url, timeout=timeout)
    except Exception:
        raise err
    finally:
        socket.getaddrinfo = orig


def kc_wait_http(url, attempts=20, interval=3, timeout=10, opener=None):
    """200 が返るまで待つ。成功したら経過秒（(i+1)*interval）、失敗なら None。"""
    opener = opener or urllib.request.urlopen
    for i in range(attempts):
        try:
            with opener(url, timeout=timeout) as r:
                if r.status == 200:
                    return (i + 1) * interval
        except Exception:
            pass
        time.sleep(interval)
    return None


def kc_retry(fn, tries=6, delay=5, skip_http_errors=True):
    """一過性の通信エラーを吸収する。HTTP 応答は「答え」なので既定で再試行しない。"""
    last = None
    for i in range(max(1, tries)):
        try:
            return fn()
        except urllib.error.HTTPError as e:
            if skip_http_errors:
                raise
            last = e
        except Exception as e:
            last = e
        if i + 1 < tries:
            wait = delay * (i + 1)
            print(f"  ⚠ 一時的な通信エラー（{i + 1}/{tries}）: {last} "
                  f"→ {wait}s 待って再試行")
            time.sleep(wait)
    raise last


def kc_post_json(url, payload, timeout=60, headers=None, opener=None):
    """JSON POST のレスポンスを返す（ストリームは呼び出し側で読む）。"""
    h = {"Content-Type": "application/json"}
    if headers:
        h.update(headers)
    req = urllib.request.Request(
        url, data=json.dumps(payload).encode(), headers=h, method="POST")
    return (opener or urllib.request.urlopen)(req, timeout=timeout)

# ── runtime/proc.py ───────────────────────────────────────
"""プロセス・GPU 状態の確認と後片付け（生成コードへ埋め込む）。"""
import re
import subprocess
import time


def _kc_live(pids, timeout=5):
    """zombie を除いた PID を返す。kill 直後は未 reap の Z が pgrep に残る。"""
    if not pids:
        return []
    try:
        out = subprocess.run(
            ["ps", "-o", "pid=,stat=", "-p", ",".join(pids)],
            capture_output=True, text=True, timeout=timeout,
        ).stdout
    except Exception:
        return pids
    live = []
    for line in out.splitlines():
        parts = line.split()
        if len(parts) >= 2 and not parts[1].startswith("Z"):
            live.append(parts[0])
    return live


def kc_proc_state(name, match="exact", timeout=5):
    """alive / DEAD / unknown。match=exact は pgrep -x、argv は pgrep -f。"""
    try:
        flag = "-x" if match == "exact" else "-f"
        out = subprocess.run(["pgrep", flag, name], capture_output=True,
                             text=True, timeout=timeout).stdout
        pids = [p for p in out.split() if p.strip()]
        return "alive" if _kc_live(pids, timeout) else "DEAD"
    except Exception:
        return "unknown"


def kc_gpu_usage(timeout=5):
    """nvidia-smi の使用率/VRAM を 1 行で返す（CPU 環境では N/A）。"""
    try:
        return subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=timeout,
        ).stdout.strip().replace("\n", " | ")
    except Exception:
        return "N/A"


def kc_vram_used(timeout=5):
    """GPU ごとの使用 VRAM (MiB)。取れなければ []。"""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=timeout,
        ).stdout
        return [int(x) for x in out.split() if x.strip().isdigit()]
    except Exception:
        return []


def kc_sample(pid, root="/proc"):
    """1 プロセスの様子を 1 つの dict にまとめる。取れない項目は None。

    `read_mib` は **実際にブロックデバイスまで行った読み込み量**（/proc の
    read_bytes）。ページキャッシュに当たった分は含まれないので、
    「重みを読み終えたか」はこちらで見る。`char_mib` は read(2) の総量で、
    キャッシュ命中も含む。両方出すのは、この 2 つが離れたときに
    「キャッシュから読んでいる」と言い切れるようにするため。

    `root` はテスト用の口。実プロセスの RSS は環境で変わるので、そこを
    assert すると「たまたま通る」テストになる。`/proc` の形をした
    ディレクトリを渡せば、変換（kB → MiB など）だけを固定できる。
    """
    s = {"rss_mib": None, "read_mib": None, "char_mib": None, "vram_mib": None}
    try:
        with open(f"{root}/{pid}/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    s["rss_mib"] = round(int(line.split()[1]) / 1024)
                    break
    except OSError:
        pass
    try:
        with open(f"{root}/{pid}/io") as f:
            for line in f:
                if line.startswith("read_bytes:"):
                    s["read_mib"] = int(line.split()[1]) // 2**20
                elif line.startswith("rchar:"):
                    s["char_mib"] = int(line.split()[1]) // 2**20
    except OSError:
        pass
    vram = kc_vram_used()
    if vram:
        s["vram_mib"] = sum(vram)
    return s


def kc_watch(pid, every=30, keep=80):
    """待機中のプロセスを `every` 秒ごとに観測する。戻り値は `(samples, stop)`。

    `stop()` を呼ぶと観測を止める。**プロセスの死だけでは止まらない**設計に
    してあるのは、待っている相手がサーバー本体だから: 起動が通れば同じ
    PID が何時間も生き続けるので、死を待つと `[watch]` が 30 秒ごとに
    何百行も出てログを埋める。呼び出し側は待機が終わった時点で止めること。

    無言の待機を「遅い」と「固まった」に分けるための計器。実測 2026-11-01
    exl3: 1800 秒の待機中に chat_server.log が **0 バイト**で、読み込みが
    進んでいたのか止まっていたのかまったく分からなかった（エンジン側の
    進捗表示は `--no-progress` で切ってあり、完走時の 1 行しか出ない）。
    そうなると次の手は「もう 1 回回す」しかなく、1 往復 30 分を捨てる。

    観測は read_bytes の**増分**が主役。1800 秒で 3 GiB しか進んでいなければ
    1.7 MiB/s ということなので、遅いのか止まったのかは即座に判る。

    戻り値の samples はそのまま result.json に入れる。失敗時にも
    「どこまで進んでいたか」が残るのが目的なので、成功・失敗を問わず入れる。
    """
    import os
    import threading

    samples = []
    stop = threading.Event()

    def loop():
        t0 = time.time()
        while not stop.wait(every):        # 止める指示が来たら即座に抜ける
            s = kc_sample(pid)
            s["t"] = round(time.time() - t0)
            samples.append(s)
            del samples[:-keep]
            print(f"  [watch] {s['t']}s rss={s['rss_mib']}MiB "
                  f"read={s['read_mib']}MiB vram={s['vram_mib']}MiB",
                  flush=True)
            try:
                os.kill(pid, 0)      # 死んでいれば ProcessLookupError
            except OSError:
                return

    threading.Thread(target=loop, daemon=True).start()
    return samples, stop.set


def kc_context_budget(vram_total_mib, model_gib, kv_kib_per_token, candidates,
                      requested=None, budget_ratio=0.88):
    """VRAM から (コンテキスト長, 予算 GiB) を決める。

    予算は実効 VRAM の `budget_ratio` 倍（T4 x2 の 30 GiB をそのまま
    使うと CUDA graph / activation で溢れるため 0.88 で見積もる）。
    `requested` が収まるならそれを優先し、駄目なら `candidates` を
    大きい順に試す。どれも収まらなければ最小候補を返す（起動は試す）。
    """
    budget_gib = vram_total_mib / 1024 * budget_ratio

    def fits(ctx):
        return model_gib + kv_kib_per_token * ctx / 1024 / 1024 <= budget_gib

    if requested and fits(requested):
        return requested, budget_gib
    for ctx in candidates:
        if fits(ctx):
            return ctx, budget_gib
    return candidates[-1], budget_gib


def kc_context_gib(model_gib, kv_kib_per_token, ctx):
    """コンテキスト 1 本ぶんの見積り（モデル + KV）を GiB で返す。"""
    return model_gib + kv_kib_per_token * ctx / 1024 / 1024


def kc_detect_gpu(timeout=15):
    """全 GPU の (名前, VRAM MiB) を返す。検出失敗は ([], [])。"""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total",
             "--format=csv,noheader"],
            capture_output=True, text=True, timeout=timeout,
        ).stdout.strip()
        names, mems = [], []
        for line in out.splitlines():
            if not line.strip():
                continue
            parts = line.split(",")
            names.append(parts[0].strip())
            mems.append(int(re.sub(r"[^0-9]", "", parts[1])))
        return names, mems
    except Exception as e:
        print(f"GPU 検出失敗: {e}")
        return [], []


def kc_wait_gpu_idle(limit_mib=500, timeout=180):
    """全 GPU の使用量が limit 未満になるまで待つ。経過秒、失敗は -1。

    名前は ollama の kc_wait_vram_free（/api/ps を見る）と衝突させない。
    """
    for i in range(timeout):
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10,
        ).stdout.strip()
        used = [int(x) for x in out.splitlines() if x.strip().isdigit()]
        if used and max(used) < limit_mib:
            return i
        time.sleep(1)
    return -1


def _kc_argv(pattern):
    """"-x cloudflared" → ["-x", "cloudflared"]（shell を介さない）。"""
    return pattern.split()


def kc_pids(pattern, timeout=15):
    """pgrep の引数文字列（例: "-x cloudflared"）に一致する PID を返す。"""
    try:
        r = subprocess.run(["pgrep", *_kc_argv(pattern)],
                           capture_output=True, text=True, timeout=timeout)
        return _kc_live([p for p in r.stdout.split() if p.strip()], timeout)
    except Exception:
        return []


def kc_stop_processes(targets, term_wait=3, final_wait=2):
    """(pgrep_pattern, label) を TERM → 待機 → KILL で止め、残存ラベルを返す。"""
    for pattern, label in targets:
        if kc_pids(pattern):
            subprocess.run(["pkill", "-TERM", *_kc_argv(pattern)],
                           capture_output=True)
            print(f"  TERM → {label}")
    time.sleep(term_wait)
    for pattern, label in targets:
        if kc_pids(pattern):
            subprocess.run(["pkill", "-KILL", *_kc_argv(pattern)],
                           capture_output=True)
            print(f"  KILL → {label}（残存していた）")
    time.sleep(final_wait)
    return [label for pattern, label in targets if kc_pids(pattern)]

# ── runtime/storage.py ────────────────────────────────────
"""検証成果物の永続化と失敗記録（生成コードへ埋め込む）。"""
import glob
import os

# `kaggle kernels output` が zip 化するのは /kaggle/working だけ。
# /output は公式の永続先ではないが、書ければ保険になるので両方へ書く。
KC_OUT_DIRS = ("/kaggle/working", "/output")


def kc_find_dataset(slug, root="/kaggle/input"):
    """Dataset のマウント先を返す（見つからなければ既定パス）。

    Kaggle は `/kaggle/input/<slug>`（旧）と
    `/kaggle/input/datasets/<owner>/<slug>`（新）が同一日内に混在する。
    深さ 2 まで探し、**空でない**ディレクトリを優先する（空マウントを
    掴むと「Dataset はあるのに中身が無い」で落ちるため）。
    呼び出し側は戻り値をそのまま使ってよい（従来の `_find_input` と同じ）。
    """
    cands = []
    for pat in (f"{root}/{slug}", f"{root}/*/{slug}", f"{root}/*/*/{slug}"):
        cands += [p for p in glob.glob(pat) if os.path.isdir(p)]
    for p in sorted(cands):
        try:
            if os.listdir(p):
                return p
        except OSError:
            continue
    return cands[0] if cands else f"{root}/{slug}"


def kc_save_output(name, text, dirs=KC_OUT_DIRS):
    """1 つでも書ければ True（成功条件に使う）。"""
    saved = []
    for d in dirs:
        try:
            os.makedirs(d, exist_ok=True)
            with open(os.path.join(d, name), "w") as f:
                f.write(text)
            saved.append(d)
        except Exception as e:
            print(f"  ⚠ {d} 書き出し失敗 ({e})")
    if saved:
        print(f"  📄 {name} 保存済: {', '.join(saved)}")
        return True
    print(f"  ⚠ 成果物を1つも保存できませんでした: {name}")
    return False


def kc_fail(msg, tail_path=None, dirs=KC_OUT_DIRS):
    """失敗原因を永続領域に残してから raise（ログ API は当てにできない）。"""
    tail = ""
    try:
        if tail_path and os.path.isfile(tail_path):
            tail = open(tail_path, errors="replace").read()[-3000:]
    except Exception:
        pass
    kc_save_output("FAILED.txt", f"{msg}\n\n{tail}", dirs)
    raise RuntimeError(msg)

# ── runtime/keepalive.py ──────────────────────────────────
"""キープアライブ設定ファイルと /stop の共通処理（生成コードへ埋め込む）。"""
import json
import os
import urllib.request


def kc_read_conf(path, default_minutes=60, max_minutes=480,
                 on_invalid="fail", fail=None, quiet_clamp=False):
    """(run, minutes, mtime) を返す。RUN=0 は (0, default_minutes, mtime)。

    on_invalid: "fail" は fail(msg)（未指定なら ValueError）、
                "zero" は停止扱い（RUN=0 と同じ）。
    quiet_clamp: proxy が書く既定値を上限で頭打ちにするとき黙る。
    """
    try:
        with open(path) as f:
            d = dict(l.strip().split("=", 1) for l in f
                     if l.strip() and "=" in l)
        run = int(d.get("RUN", 1))
        minutes = int(d.get("MINUTES", default_minutes))
    except OSError:
        run, minutes = 1, default_minutes
    except ValueError:
        run, minutes = 1, default_minutes
    try:
        mtime = os.path.getmtime(path)
    except OSError:
        mtime = None
    if run <= 0:
        return 0, default_minutes, mtime
    if minutes <= 0:
        msg = f"KEEPALIVE_MINUTES={minutes} は無効（1〜{max_minutes} で指定）"
        if on_invalid == "zero":
            return 0, default_minutes, mtime
        if fail:
            fail(msg)
        raise ValueError(msg)
    if minutes > max_minutes:
        if not (quiet_clamp and minutes == default_minutes):
            print(f"⚠ KEEPALIVE_MINUTES={minutes} → {max_minutes} に制限")
        minutes = max_minutes
    return run, minutes, mtime


def kc_write_conf(path, run, minutes=None):
    with open(path, "w") as f:
        f.write(f"RUN={run}\n")
        if minutes is not None:
            f.write(f"MINUTES={minutes}\n")


def kc_conf_mtime(path):
    """keepalive.conf の更新時刻。proxy がチャット送信ごとに書き直す。"""
    try:
        return os.path.getmtime(path)
    except OSError:
        return 0.0


def kc_keepalive_deadline(started, minutes, mtime, hard_max_minutes=180):
    """停止時刻。最後の活動（起動 or conf 更新）から minutes 分、全体は hard_max で頭打ち。"""
    last = max(started, mtime or started)
    return min(last + minutes * 60, started + hard_max_minutes * 60)


def kc_keepalive_expired(now, started, minutes, mtime, hard_max_minutes=180):
    """now 時点で時間予算を超えたか（now を渡すのでフェイククロックで検証できる）。"""
    return now >= kc_keepalive_deadline(started, minutes, mtime,
                                        hard_max_minutes)


def kc_stop_via_proxy(port, auth=None, timeout=30):
    headers = {"X-Kc-Stop": "1"}
    if auth:
        headers["Authorization"] = auth
    req = urllib.request.Request(
        f"http://127.0.0.1:{port}/stop", data=b"", method="POST",
        headers=headers)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.status, json.loads(r.read())

# ── runtime/proxy.py ──────────────────────────────────────
"""auth_proxy の起動と認証なし検査（生成コードへ埋め込む）。"""
import subprocess
import time
import urllib.error
import urllib.request


def kc_start_proxy(port, upstream, chat_html, proxy_path,
                   extra_args=(), env=None, log_path="/tmp/auth_proxy.log",
                   wait_seconds=30, label="プロキシ", fail=None,
                   python="python3"):
    """auth_proxy.py を起動し /healthz を待つ。失敗時は fail() か RuntimeError。"""
    subprocess.run(["pkill", "-f", "auth_proxy.py"], capture_output=True)
    time.sleep(1)
    cmd = [python, proxy_path, "--port", str(port),
           "--upstream", upstream, "--chat-html", chat_html, *extra_args]
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, env=env, stdout=log, stderr=log)
    print(f"🔄 proxy PID: {proc.pid}")

    elapsed = None
    for i in range(wait_seconds):
        try:
            with urllib.request.urlopen(
                    f"http://127.0.0.1:{port}/healthz", timeout=3) as r:
                if r.status == 200:
                    elapsed = i + 1
                    break
        except Exception:
            pass
        time.sleep(1)
    if elapsed is None:
        try:
            print(open(log_path).read()[-2000:])
        except OSError:
            pass
        msg = f"{label}の起動に失敗しました（{log_path}）"
        if fail:
            fail(msg, log_path)
        raise RuntimeError(msg)
    print(f"✅ {label}起動完了（{elapsed}s）")
    return proc


def kc_assert_no_auth(port, path="/api/tags", fail=None, timeout=10):
    """401/403 なら認証が有効 = 設定ミス。上流未起動の 5xx は正常。"""
    req = urllib.request.Request(f"http://127.0.0.1:{port}{path}")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            print(f"  ✅ 認証なし → HTTP {r.status}")
            return True
    except urllib.error.HTTPError as e:
        if e.code in (401, 403):
            msg = f"認証なしで HTTP {e.code} — 認証が有効になっています"
            if fail:
                fail(msg)
            raise RuntimeError(msg)
        print(f"  ✅ 認証なし → HTTP {e.code}（401 でない）")
        return True
    except Exception as e:
        msg = f"プロキシ検査に失敗: {type(e).__name__}: {e}"
        if fail:
            fail(msg)
        raise

# ── runtime/beacon.py ─────────────────────────────────────
"""ビーコンカーネル（Kaggle のカーネルを掲示板として使う）。

**カーネルのコードそのもの**を公開 URL の置き場にする。

- 書き手: 配信カーネルの中から `kernels_push` で小さなカーネルを更新する。
  認証は Kaggle がカーネルへ注入するもの（Secrets 不要）。
- 読み手: `kaggle kernels pull` でコードを引いて `URL` を読む。

外部サービスに URL を置く方式をやめた理由: Kaggle の Secret は**カーネル
ごとに紐付ける**必要があり、未紐付けの `get_secret()` は `ConnectionError`
を返す（通信障害に見える）。専用アカウント運用では、カーネルを増やすたびに
UI 作業が要り、トークンも個人アカウントのものが残る。ビーコンは
**アカウント・Secrets・外部サービスを増やさずに**同じことができる。
経緯は `notes/url-publishing.md`。

この層の方針: **失敗しても配信を止めない。ただし理由は必ず返す。**
外部の置き場が 3 日間更新されていないことに誰も気づかない、という失敗を
しているので、`(ok, detail)` を返さない関数はここに置かない。

`kc_` 接頭辞は生成コードの名前空間で衝突しないため（runtime/__init__.py）。
"""
import json
import os
import shutil
import sys
import tempfile
import time

# 掲示板の中身。**この文字列がそのまま beacon カーネルの main.py になる。**
# 値は repr で埋めるので、URL に引用符が混ざっても壊れない。
_BEACON_BODY = '''"""公開 URL のビーコン（掲示板）。

このカーネルは**実行しない**。コードそのものが受け渡しの場所で、
配信カーネルが `kernels_push` でこのファイルを書き換える。
読む側は `kaggle kernels pull` して `URL` と `UPDATED` を読む。

`STAGE` は `serving` / `probe` など。**`probe` は本物ではない**ので、
読む側は `probe` を捨てること（テスト URL を掴む事故を防ぐ）。
`UPDATED` は発行時刻の epoch 秒。古い値を掴んでいないかの判定に使う。
"""

URL = {url!r}
STAGE = {stage!r}
UPDATED = {updated!r}
'''


def kc_beacon_slug(slug=""):
    """`owner/name` を解決する（引数 → 環境変数）。無ければ空文字。"""
    s = (slug or "").strip()
    if s:
        return s
    return (os.environ.get("KC_BEACON_SLUG", "") or "").strip()


def kc_kaggle_path(entries=None):
    """`import kaggle` を壊すパスを取り除いた `sys.path` を返す。

    リポジトリ直下には `kaggle/`（カーネルのビルド元）がある。これが
    **インストール済みの `kaggle` パッケージを隠す**ので、ローカルで
    `cli beacon` を叩くと `No module named 'kaggle.api'` で死ぬ（実測）。
    カーネル内には `kaggle/` ディレクトリが無いので、ここは素通りする。

    判定は「`kaggle/` があり、かつ `kaggle/api` が無い」パスだけを外す。
    `kaggle/api` を持つ本物の候補は残すので、site-packages は消えない。
    """
    src = list(sys.path if entries is None else entries)
    keep = []
    for p in src:
        base = p or "."
        try:
            shadow = (os.path.isdir(os.path.join(base, "kaggle"))
                      and not os.path.isdir(os.path.join(base, "kaggle", "api")))
        except OSError:                       # 権限の無いパスは触らない
            shadow = False
        if not shadow:
            keep.append(p)
    return keep


def kc_kaggle_api():
    """`KaggleApi` クラスを返す。無ければ例外を投げる（呼び出し側が拾う）。

    普通に import して駄目だったときだけ `kc_kaggle_path` で探し直す。
    一度失敗した `kaggle` は名前空間パッケージとして `sys.modules` に
    残っているので、それも捨てないと再試行が同じ場所へ行く。
    """
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        return KaggleApi
    except Exception:
        saved = list(sys.path)
        try:
            sys.path[:] = kc_kaggle_path(saved)
            sys.modules.pop("kaggle", None)
            sys.modules.pop("kaggle.api", None)
            from kaggle.api.kaggle_api_extended import KaggleApi
            return KaggleApi
        finally:
            sys.path[:] = saved


def kc_beacon_meta(slug):
    """beacon カーネルの kernel-metadata.json を組み立てる。

    **実行しない掲示板**なので GPU もインターネットも要らない。非公開にするのは、
    URL が「知っていれば誰でも入れる入口」だから（`url.txt` と同じ扱い）。
    """
    name = slug.rsplit("/", 1)[-1]
    return {
        "id": slug,
        "title": name,
        "code_file": "main.py",
        "language": "python",
        "kernel_type": "script",
        "is_private": True,
        "enable_gpu": False,
        "enable_tpu": False,
        "enable_internet": False,
        "keywords": [],
        "dataset_sources": [],
        "kernel_sources": [],
        "competition_sources": [],
        "model_sources": [],
    }


def kc_beacon_body(url, stage="", updated=None):
    """掲示板の main.py の中身。"""
    return _BEACON_BODY.format(
        url=url, stage=stage, updated=time.time() if updated is None else updated)


def kc_beacon_publish(url, stage="", slug="", timeout=180):
    """掲示板を更新する。戻り値は `(ok, detail)`。

    `detail` は成功なら `"pushed v3 (platform)"` のような内訳、失敗なら理由。
    **例外は投げない**（配信を止めない）。呼び出し側は成否を `result` に残す。
    """
    s = kc_beacon_slug(slug)
    if not s:
        return False, "beacon の slug が無い（KC_BEACON_SLUG か引数で渡す）"
    if "/" not in s:
        return False, f"slug は owner/name の形で渡す: {s!r}"
    try:
        KaggleApi = kc_kaggle_api()
    except Exception as e:
        return False, f"kaggle パッケージが無い（{type(e).__name__}: {e}）"

    d = tempfile.mkdtemp(prefix="beacon-")
    try:
        with open(os.path.join(d, "main.py"), "w", encoding="utf-8") as f:
            f.write(kc_beacon_body(url, stage))
        with open(os.path.join(d, "kernel-metadata.json"), "w",
                  encoding="utf-8") as f:
            json.dump(kc_beacon_meta(s), f, indent=2, ensure_ascii=False)

        api = KaggleApi()
        try:
            api.authenticate()
        except Exception as e:
            return False, (f"Kaggle 認証が無い（{type(e).__name__}: {e}）"
                           " — カーネル外からは ~/.kaggle/kaggle.json が要る")
        try:
            resp = api.kernels_push(d)
        except SystemExit as e:                      # kaggle は終了コードで返す
            return False, f"kernels_push が終了した（SystemExit {e.code}）"
        except Exception as e:
            return False, f"kernels_push 失敗（{type(e).__name__}: {e}）"
        ver = getattr(resp, "version_number", None)
        return True, f"pushed v{ver} ({s})"
    finally:
        shutil.rmtree(d, ignore_errors=True)


def kc_beacon_parse(text):
    """掲示板のソースから `url` / `stage` / `updated` を取り出す。

    `exec` しない。中身は自分で書いたものだが、読む側（VM）にとっては
    ネットワーク越しの入力なので、評価せずに読む。
    """
    out = {"url": "", "stage": "", "updated": None}
    keys = {"URL": "url", "STAGE": "stage", "UPDATED": "updated"}
    for line in (text or "").splitlines():
        s = line.strip()
        for key, field in keys.items():
            if not s.startswith(key + " ="):
                continue
            raw = s.split("=", 1)[1].strip()
            if field == "updated":
                try:
                    out[field] = float(raw)
                except ValueError:
                    out[field] = None
            else:
                out[field] = raw.strip("\"'")
    return out


def kc_beacon_is_fresh(data, since, slack=5.0):
    """`since` より後に発行されたものだけを本物とみなす。

    2026-11-01 に**古いテスト URL を掴む**事故が出ている。push の前後で
    スタンプが変わったことを確かめないと、「まだ前の値」を新しい値として
    読んでしまう。`slack` は VM と Kaggle の時計のずれぶん。
    """
    u = (data or {}).get("updated")
    if not isinstance(u, (int, float)):
        return False
    return u >= (float(since) - float(slack))


def kc_beacon_pull_cli(slug, dest, exe="", timeout=180):
    """`kaggle kernels pull` を**CLI で**実行する。戻り値は `(ok, detail)`。

    Python パッケージの `kaggle` が import できない環境のための逃げ道。
    手元の python3（3.13）には入っておらず、`kaggle` CLI だけが 3.9 の
    user site に入っている、という状態で実際に踏んだ（`cli beacon` が
    `No module named 'kaggle'` で死んでいた）。カーネル内では使われない。
    """
    import subprocess                      # 束ねたときの F811 を避けて局所 import
    cmd = [exe or shutil.which("kaggle") or "", "kernels", "pull", slug, "-p", dest]
    if not cmd[0]:
        return False, "kaggle パッケージも kaggle CLI も見つからない"
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except Exception as e:
        return False, f"kaggle CLI の起動に失敗（{type(e).__name__}: {e}）"
    if p.returncode != 0:
        tail = (p.stderr or p.stdout or "").strip().splitlines()[-1:] or [""]
        return False, f"kaggle CLI が {p.returncode} で終了（{tail[0][:200]}）"
    return True, "kaggle CLI"


def kc_beacon_pull(slug="", dest=None, timeout=180, exe=""):
    """掲示板を引いて読む。戻り値は `(ok, detail, data)`。

    `data` は `kc_beacon_parse` の結果（失敗なら空）。
    `exe` は CLI 経由に落ちたときに使う `kaggle` のパス（`cli push` と同じ
    解決順を呼び出し側が渡す）。空なら PATH から探す。
    """
    s = kc_beacon_slug(slug)
    if not s:
        return False, "beacon の slug が無い", {}
    d = dest or tempfile.mkdtemp(prefix="beacon-pull-")
    try:
        # まず Python パッケージ。駄目なら CLI に落ちる（理由は捨てない）。
        why_api = ""
        try:
            KaggleApi = kc_kaggle_api()
        except Exception as e:
            why_api = f"{type(e).__name__}: {e}"
        else:
            try:
                api = KaggleApi()
                api.authenticate()
                api.kernels_pull(s, path=d)
                why_api = ""
            except SystemExit as e:          # kaggle は終了コードで返す
                why_api = f"kernels_pull が終了した（SystemExit {e.code}）"
            except Exception as e:
                why_api = f"kernels_pull 失敗（{type(e).__name__}: {e}）"
        if why_api:
            ok, detail = kc_beacon_pull_cli(s, d, exe=exe, timeout=timeout)
            if not ok:
                return False, f"API でも CLI でも引けなかった（API: {why_api} / CLI: {detail}）", {}
        # 引いたファイル名は版によって変わる（main.py / <name>.py）ので、
        # 中身で選ぶ。UPDATED を含む .py が掲示板。
        for root, _dirs, files in os.walk(d):
            for fn in files:
                if not fn.endswith(".py"):
                    continue
                with open(os.path.join(root, fn), encoding="utf-8",
                          errors="replace") as f:
                    text = f.read()
                if "UPDATED" in text:
                    data = kc_beacon_parse(text)
                    return True, f"pulled {fn}", data
        return False, "pull した中に掲示板（UPDATED を含む .py）が無い", {}
    finally:
        if dest is None:
            shutil.rmtree(d, ignore_errors=True)



---
### ⚙️ 設定（GPU検出 → モデル・コンテキスト決定）


In [ ]:
import os
os.environ.setdefault('VLLM_SAFETENSORS_STRATEGY', 'prefetch')
os.environ.setdefault('VLLM_SAFETENSORS_PREFETCH_THREADS', '1')
os.environ.setdefault('KC_PREFETCH_PATCH', '1')
os.environ.setdefault('VLLM_STARTUP_BUDGET', '11000')
os.environ.setdefault('VLLM_ATTEMPT_TIMEOUT', '11000')
os.environ.setdefault('KC_PROBE_GATE_MIB_S', '150')
os.environ.setdefault('KC_PROBE_NTFY_TOPIC', '')
os.environ.setdefault('KC_PROBE_SIDE', '')

# ============================================================
# Step 2 (vLLM): 配信カーネルの設定
# ============================================================
import json
import os
import subprocess

TIMINGS_PATH = "/kaggle/working/timings.json"


def kc_save_timings(d):
    """計測値を timings.json に足す（result.json へはテンプレートの save() が合流）。

    セルから `data` を直接触らないのは、**notebook 版には `data` が無い**ため
    （セル順の「定義前使用」で落ちる。2026-11-01 実測）。ファイル経由なら
    notebook / script の両方で同じコードが動く。

    計測は「高度化の基礎」なので、失敗しても本処理は止めない。
    """
    try:
        with open(TIMINGS_PATH, encoding="utf-8") as f:
            cur = json.load(f)
    except (OSError, ValueError):
        cur = {}
    cur.update(d)
    try:
        with open(TIMINGS_PATH, "w", encoding="utf-8") as f:
            json.dump(cur, f, ensure_ascii=False, indent=2)
    except OSError as _e:
        print(f"⚠ timings.json を書けません: {_e}")

# Dataset のマウント先は共通ランタイム（runtime/storage.py）に一本化。
# 旧 /kaggle/input/<slug> と新 /kaggle/input/datasets/<owner>/<slug> の
# 両対応・空マウント回避は kc_find_dataset が持つ。
MODEL_SRC = kc_find_dataset("qwen38-27b-uncensored-hauhaucs-awq-mtp")
# 共有セルは MODEL_NAME を読む。vLLM の文脈では served-model-name と同義。
SERVED_NAME = MODEL_NAME = "qwen3.8-27b-awq"
MODEL_DISPLAY = "Qwen3.8-27B-Uncensored (AWQ)"
ENGINE_DISPLAY = "vLLM"
ENGINE_LABEL = "vllm"

# ---- ポート ----
# vLLM 本体は 8000、公開側はプロキシが 11435 で受ける。
PORT = UPSTREAM_PORT = VLLM_PORT = 8000
PROXY_PORT = 11435
PROXY_CHECK_PATH = "/v1/models"
# クライアントが temperature=0 / repetition_penalty 未指定で送っても
# 崩壊しないよう、サーバ側で下限を矯正する（RP>=1.05 / temp>0）。
PROXY_EXTRA_ARGS = ("--enforce-sampling",)
TUNNEL_KIND = "cloudflared"

# Basic 認証プロキシ / UI のソース（ビルダーが埋め込む）
PROXY_SRC_B64 = "..."
CHAT_SRC_B64 = "..."
PROXY_PATH = "/usr/local/bin/auth_proxy.py"
CHAT_PATH = "/usr/local/bin/llama-chat.html"

# pip に渡す vLLM 指定（切り戻したいときは "vllm==0.10.2" 等に変える）
VLLM_PIP_SPEC = os.environ.get("VLLM_PIP_SPEC", "vllm==0.32.0")


_gpu_names, _vram_each = kc_detect_gpu()
VRAM_TOTAL = sum(_vram_each)
print(f"GPU      : {_gpu_names}")
print(f"VRAM     : {_vram_each} MiB each  →  合計 {VRAM_TOTAL} MiB")

# /dev/shm は TP>1 の必須資源。vLLM のリングバッファは既定で
# (24 MiB + 数 B) x 10 ブロック ≈ 240 MiB を 1 本で確保する。
# 足りなければ即 RuntimeError になるが、**境界値だと起動後に詰まる**（実測:
# 32K ctx で "No available shared memory broadcast block found" が 60s 間隔で連発）。
def _shm_required_mib(tp):
    """/dev/shm の必要量（MiB）の目安。TP 本ぶん + 2 倍の余裕。

    1 本あたり 24 MiB x 10 ブロック ≒ 240 MiB。TP=2 なら 480 MiB、
    余裕を 2 倍見て 960 MiB。**空き**で判定する（total では足りない。
    他プロセスの残骸が掴んでいる場合がある）。
    """
    return 240 * max(1, int(tp)) * 2


SHM_FREE_MIB = SHM_TOTAL_MIB = 0
try:
    _shm = os.statvfs("/dev/shm")
    SHM_FREE_MIB = _shm.f_bavail * _shm.f_frsize // (1 << 20)
    SHM_TOTAL_MIB = _shm.f_blocks * _shm.f_frsize // (1 << 20)
    print(f"/dev/shm : {SHM_TOTAL_MIB} MiB（空き {SHM_FREE_MIB} MiB）"
          f"  cpu={os.cpu_count()}")
except Exception as _e:
    print(f"/dev/shm : 取得失敗 {_e!r}")

if VRAM_TOTAL == 0:
    raise RuntimeError(
        "GPU が見つかりません。Accelerator を 'GPU T4 x2' に設定してください。")
if len(_vram_each) < 2:
    print("⚠ GPU が 1 枚です。Tensor Parallel は 1 に落とします"
          "（27B AWQ は 1 枚に収まらない可能性が高い）")
TP_SIZE = 2 if len(_vram_each) >= 2 else 1

# 必要量は TP が決まってから判定する（TP に比例するため）。
SHM_REQUIRED_MIB = _shm_required_mib(TP_SIZE)
if SHM_TOTAL_MIB and SHM_FREE_MIB < SHM_REQUIRED_MIB:
    print(f"⚠ /dev/shm の空き {SHM_FREE_MIB} MiB < 必要 {SHM_REQUIRED_MIB} MiB"
          f"（TP={TP_SIZE}）→ 拡張を試みる")
    try:
        _rm = subprocess.run(
            ["mount", "-o", "remount,size=8G", "/dev/shm"],
            capture_output=True, text=True, timeout=60)
        print(f"   remount rc={_rm.returncode} {( _rm.stderr or '').strip()[:120]}")
    except Exception as _e:
        print(f"   remount 失敗: {_e!r}")
    try:
        _shm = os.statvfs("/dev/shm")
        SHM_FREE_MIB = _shm.f_bavail * _shm.f_frsize // (1 << 20)
        SHM_TOTAL_MIB = _shm.f_blocks * _shm.f_frsize // (1 << 20)
        print(f"/dev/shm : 再測定 {SHM_TOTAL_MIB} MiB（空き {SHM_FREE_MIB} MiB）")
    except Exception as _e:
        print(f"/dev/shm : 再測定失敗 {_e!r}")
    if SHM_FREE_MIB < SHM_REQUIRED_MIB:
        # ここで落とす。放置すると起動はするが forward で詰まり、
        # 待機側が 40 分待つ（1 往復を丸損する）。
        raise RuntimeError(
            f"/dev/shm が不足しています（空き {SHM_FREE_MIB} MiB / 必要 "
            f"{SHM_REQUIRED_MIB} MiB, TP={TP_SIZE}）。TP=1 に落とすか、"
            "max-model-len を下げてください。")

# ---- 起動条件 ----
# 27B AWQ は約 19 GB。T4 x2 = 30 GB のうち 0.90 = 27 GB を使う想定で、
# 残り ~8 GB が KV cache になる。実際に収まるかは config 次第なので、
# 起動失敗時に max-model-len を落として再試行する（serve セル）。
# 2026-10-29 机上計算: 重み 18.20 GiB + KV f16 68 KiB/token。
# 64K=22.45 GiB（予算 27.0 で余裕 4.5）、96K=24.58（余裕 2.4）、
# 128K=26.70（余裕 0.3 = graph/activation で確実に溢れる）。
# T4 は fp8 KV 不可なので Ollama の q8_0（34 KiB/token）のようには詰められない。
# → 上限狙いは 64K。実測で 1〜2 GiB ずれるので 32K/16K も残す。
MAXLEN_CANDIDATES = [65536, 32768, 16384, 8192]
MAX_MODEL_LEN = MAXLEN_CANDIDATES[0]
# 共有セル（接続情報）が読む。起動成功後は実際の max_model_len で上書きする。
CONTEXT_LENGTH = MAX_MODEL_LEN
GPU_MEM_UTIL = 0.90
MAX_NUM_SEQS = 2
KV_CACHE_DTYPE = "auto"        # T4 は FP8 非対応。fp8 を指定してはいけない

# MTP（multi-token prediction）は任意。失敗したら自動で無効にして再起動する。
MTP_ENABLED = os.environ.get("VLLM_MTP", "1") != "0"
MTP_METHOD = os.environ.get("VLLM_SPEC_METHOD", "mtp")
MTP_TOKENS = int(os.environ.get("VLLM_SPEC_TOKENS", "2"))

# ---- 認証 ----
# 既定で無効。API_PASS が入っているときだけ proxy が有効化する。
PROXY_AUTH = {
    "API_USER": os.environ.get("API_USER", "") or "ollama",
    "API_PASS": os.environ.get("API_PASS", ""),
}

# ---- アカウント（ビーコンの slug に使う。config.env の KAGGLE_OWNER）----
KAGGLE_OWNER = "yamahei21python"

# ---- キープアライブ ----
# 上限。チャット 1 回で proxy が MINUTES=60 を書くため、上限が無いと
# 「1 回送信 → 60 分に延長」で残クォータを超えて強制終了される
# （強制終了は成果物の zip が保証されない）。検証中は 30 分で頭打ちにする。
MAX_MINUTES = 30
KEEPALIVE_RESET_MINUTES = 60      # チャット送信ごとにリセットする時間（分）
TUNNEL_CHECK_EVERY = 10           # 公開URL疎通は 10 分ごと

# ---- 死活監視 / 後片付けの能力表（共有セルが読む）----
ENGINE_LOG = "/tmp/vllm_1.log"
ENGINE_PROCS = [("vllm serve", "argv"), ("auth_proxy.py", "argv")]
CLEANUP_TARGETS = [
    ("-x cloudflared", "cloudflared"),
    ("-f auth_proxy.py", "auth_proxy"),
    ("-f vllm serve", "vllm"),
    ("-f VLLM::EngineCore", "engine-core"),
]


def engine_cleanup():
    """次回起動を短くするため compile / Triton キャッシュを退避する。

    VRAM の解放はプロセス停止（CLEANUP_TARGETS が EngineCore まで落とす）で行う。
    3GB を超える場合は退避しない（転送と容量の無駄）。
    """
    try:
        import tarfile as _tf
        keep = []
        for _d in ("/root/.cache/vllm", "/root/.triton", "/root/.cache/triton"):
            if os.path.isdir(_d) and not any(_d.startswith(k + "/") for k in keep):
                keep.append(_d)
        size = 0
        for _d in keep:
            for _root, _dirs, _files in os.walk(_d):
                for _f in _files:
                    try:
                        size += os.path.getsize(os.path.join(_root, _f))
                    except OSError:
                        pass
        print(f"🧠 キャッシュ {size/1e6:.0f} MB {keep}")
        if keep and size and size < 3e9:
            out = os.path.join(OUT_DIRS[0], "warm-cache.tar")
            with _tf.open(out, "w") as _t:
                for _d in keep:
                    _t.add(_d, arcname=os.path.relpath(_d, "/"))
            print(f"   退避: {out} {os.path.getsize(out)/1e6:.0f} MB")
            return f"cache_tar={os.path.getsize(out)}"
        return "cache_skipped" if size else "cache_none"
    except Exception as _e:
        print(f"⚠ キャッシュ退避をスキップ: {_e}")
        return f"cache_error={type(_e).__name__}"


# ---- 検証成果物の永続化 ----
# `kaggle kernels output` が zip 化するのは /kaggle/working のみ。
OUT_DIRS = KC_OUT_DIRS
save_output = kc_save_output
fail = kc_fail


print()
print("=" * 56)
print(f"  モデル      : {MODEL_SRC}")
print(f"  配信名      : {SERVED_NAME}")
print(f"  TP          : {TP_SIZE}")
print(f"  MTP         : {'ON' if MTP_ENABLED else 'OFF'}")
print(f"  公開ポート  : {PROXY_PORT}（vLLM 本体は {VLLM_PORT}）")
print(f"  認証        : {'ON' if PROXY_AUTH['API_PASS'] else 'OFF（URL を知れば誰でも利用可）'}")
print("=" * 56)

_files = sorted(os.listdir(MODEL_SRC)) if os.path.isdir(MODEL_SRC) else []
print(f"  Dataset 直下: {len(_files)} 件")
for _f in _files[:12]:
    print(f"    - {_f}")
if "config.json" not in _files:
    fail("config.json が Dataset 直下にありません（Add Data を確認）")
if "model.safetensors" not in _files:
    fail("model.safetensors が Dataset 直下にありません（Add Data を確認）")


# ---- 起動前の I/O probe（ノードの実効速度を測る）--------------------
# 同じコードでも重みロードが 1,304〜6,128 秒と 4.7 倍振れる（notes §25）。
# 差の大半は**ノードの GCSFUSE 実効速度**で、こちらからは選べない。
# そこで起動前に 64 MiB を 1 本で逐次読みし、19.0 GiB を割って予算に
# 収まるかを見る。遅いノードを引いた回に 100 分焼かず、30 秒で諦めるため。
#
# `KC_PROBE_ONLY=1` なら**計測だけして終了**する（install の 270 秒も払わない）。
# 10 回まわして「速いノード / 遅いノード」をどれだけ弾けるかの実測に使う。
def _kc_probe_safetensors():
    """Dataset 直下の `.safetensors` を (合計バイト, 最大ファイル) で返す。"""
    total, biggest = 0, (0, "")
    try:
        names = sorted(os.listdir(MODEL_SRC))
    except OSError:
        return 0, (0, "")
    for fn in names:
        if not fn.endswith(".safetensors"):
            continue
        p = os.path.join(MODEL_SRC, fn)
        try:
            size = os.path.getsize(p)
        except OSError:
            continue
        total += size
        if size > biggest[0]:
            biggest = (size, p)
    return total, biggest


def _kc_probe_io(mib=64):
    """最大の safetensors の末尾を `mib` MiB 読んで実効速度を返す（1 本）。"""
    import time

    total, (size, path) = _kc_probe_safetensors()
    if not path:
        return {"ok": False, "why": "safetensors が無い"}
    n = min(int(mib) * 2**20, size)
    off = max(0, size - n)                 # 末尾（本読みと窓をずらす）
    t0 = time.time()
    try:
        fd = os.open(path, os.O_RDONLY)
        try:
            got = len(os.pread(fd, n, off))
        finally:
            os.close(fd)
    except OSError as e:
        return {"ok": False, "why": f"{type(e).__name__}: {e}"}
    dt = time.time() - t0
    if not got or dt <= 0:
        return {"ok": False, "why": "0 バイト"}
    rate = got / 2**20 / dt
    projected = total / 2**20 / rate
    limit = int(os.environ.get("KC_PROBE_LIMIT_S", "6400"))
    return {
        "ok": True,
        "file": os.path.basename(path),
        "read_mib": round(got / 2**20, 1),
        "seconds": round(dt, 3),
        "mib_s": round(rate, 2),
        "total_mib": round(total / 2**20, 1),
        "projected_s": round(projected),
        # 重みロード以外（install / compile / capture）に約 900 秒みる
        "verdict": "pass" if projected + 900 <= limit else "reject",
        "limit_s": limit,
        "mem_available_mib": None,
    }


def _kc_probe_report(rec):
    if not rec.get("ok"):
        print(f"🔎 probe 失敗: {rec.get('why')}")
        return
    print(f"🔎 ノード probe: {rec['read_mib']} MiB / {rec['seconds']}s = "
          f"**{rec['mib_s']} MiB/s**")
    print(f"   19.0 GiB 相当 {rec['total_mib']} MiB → 重み {rec['projected_s']} 秒"
          f"（上限 {rec['limit_s']} 秒 + 固定 900 秒） → {rec['verdict'].upper()}")
    print("PROBE_JSON " + json.dumps(rec, ensure_ascii=False))


def _kc_probe_beacon(rec):
    """probe 結果を ntfy へ 1 行 POST する（レース用・best effort）。

    `kaggle kernels output` は**実行が終わるまで**取れない（実測）。レースでは
    勝ち側が install 以降で何分も走るので、probe の時点で外から知る手段が
    要る。失敗しても本番の邪魔をしない（例外は握る）。
    """
    topic = os.environ.get("KC_PROBE_NTFY_TOPIC", "").strip()
    if not topic or not rec.get("ok"):
        return
    body = ("mib_s=%s side=%s" % (
        rec.get("mib_s"), os.environ.get("KC_PROBE_SIDE", ""))).strip()
    try:
        import urllib.request
        urllib.request.urlopen(urllib.request.Request(
            "https://ntfy.sh/" + topic, data=body.encode(), method="POST"),
            timeout=8).read()
    except Exception as _e:
        print(f"⚠ beacon 送信失敗: {type(_e).__name__}: {_e}")


def _kc_probe_gate(rec, gate):
    """probe 結果がゲートを満たすか。`gate <= 0` は常に通す。

    判定に使うのは **1 本読みの速度だけ**。重みロード速度の予測ではない
    （notes §28: 相関は未検証）。install（267 秒）に入る前に降りるための
    足切りで、遅いノードを引いた回に 100 分焼かないためのもの。
    probe が失敗した回は判定できないので通す（安全側 = 走らせる）。
    """
    if gate <= 0 or not rec.get("ok"):
        return True
    return rec["mib_s"] >= gate


def _kc_probe_mem_mib():
    """`MemAvailable`（MiB）。03 の `_mem_info` は**まだ定義されていない**ので使わない。"""
    try:
        with open("/proc/meminfo", encoding="utf-8") as _mf:
            for _ln in _mf:
                if _ln.startswith("MemAvailable:"):
                    return int(_ln.split()[1]) // 1024
    except (OSError, ValueError, IndexError):
        pass
    return None


_KC_PROBE = _kc_probe_io()
_KC_PROBE["mem_available_mib"] = _kc_probe_mem_mib()
_kc_probe_report(_KC_PROBE)
_kc_probe_beacon(_KC_PROBE)
kc_save_timings({"io_probe": _KC_PROBE})
try:
    with open("/kaggle/working/probe.json", "w", encoding="utf-8") as _f:
        json.dump(_KC_PROBE, _f, ensure_ascii=False, indent=2)
except OSError as _e:
    print(f"⚠ probe.json を書けません: {_e}")

if os.environ.get("KC_PROBE_ONLY", "").strip().lower() not in ("", "0", "false", "no"):
    print("⏹ KC_PROBE_ONLY=1 → 計測だけして終了（install 以降は走らせない）")
    raise SystemExit(0)

# ---- ノード速度ゲート（`cli race` 用）--------------------------------
# `KC_PROBE_GATE_MIB_S` 未満のノードは **install に入る前**に降りる。
# 2 本同時に push し、速いノードを引くまで繰り返すレースの土台。
# 0 / 未設定なら無効（従来どおり全部走る）。
_KC_GATE = float(os.environ.get("KC_PROBE_GATE_MIB_S", "0") or 0)
if _KC_GATE > 0:
    _kc_gate_pass = _kc_probe_gate(_KC_PROBE, _KC_GATE)
    kc_save_timings({"probe_gate": {
        "gate_mib_s": _KC_GATE,
        "mib_s": _KC_PROBE.get("mib_s"),
        "pass": _kc_gate_pass,
    }})
    print(f"🚦 ノードゲート: {_KC_PROBE.get('mib_s')} MiB/s "
          f"{'≥' if _kc_gate_pass else '<'} {_KC_GATE} → "
          f"{'続行' if _kc_gate_pass else '早期終了（install 前）'}")
    if not _kc_gate_pass:
        raise SystemExit(0)


---
### 📡 vLLM インストール + プロキシ配置

vLLM の wheel 取得に 5〜10 分かかります（2回目以降は Kaggle のキャッシュで短縮）。


In [ ]:
# vLLM をインストールし、プロキシと UI を書き出す
import base64
import glob as _glob
import os
import shutil
import subprocess
import sys
import time

BIN_DIR = "/usr/local/bin"
MIN_SIZE = 1024 * 1024

# ---- 計測（高度化の基礎）----
# result.json の `timings.install_seconds` は **None のまま**だった。vLLM は
# install をテンプレートの `stage_install()` ではなくこのセルで行うため、
# 誰も記録していなかった。比較できないと「速くなった/遅くなった」を言えない。
# 区間の**絶対オフセット**（セル開始からの秒）を残す。差分は後から計算できる。
_T_INSTALL = time.time()
_MARKS = {}


def _mark(name):
    """区間の到達時刻を記録する（セル開始からの秒）。"""
    _MARKS[name] = round(time.time() - _T_INSTALL, 1)
    return _MARKS[name]

# ---- cloudflared（Dataset キャッシュ優先）----
# トンネルは cloudflared だけなので、種類で分岐しない。常に用意する。
CF_PATH = kc_find_dataset("my-cloudflared-bin")
dst = os.path.join(BIN_DIR, "cloudflared")
src = None
for cand in (os.path.join(CF_PATH, "cloudflared"),
             os.path.join(CF_PATH, "bin", "cloudflared")):
    if os.path.isfile(cand) and os.path.getsize(cand) >= MIN_SIZE:
        src = cand
        break
if src:
    shutil.copy2(src, dst)
    subprocess.run(["chmod", "+x", dst])
    print(f"✅ cloudflared 復元: {src}")
elif not (os.path.isfile(dst) and os.path.getsize(dst) >= MIN_SIZE):
    print("⬇️  cloudflared をダウンロード中...")
    subprocess.run(
        "curl -fsSL --connect-timeout 15 -o /usr/local/bin/cloudflared "
        f"{kc_cloudflared_url()}"
        " && chmod +x /usr/local/bin/cloudflared",
        shell=True, check=False, timeout=300,
    )
if not os.path.isfile(dst):
    fail("cloudflared がありません（Dataset 未マウント / 取得失敗）")
r = subprocess.run([dst, "--version"], capture_output=True, text=True)
print((r.stdout or r.stderr).strip())
if r.returncode != 0:
    fail("cloudflared の実行に失敗しました（実行ビット/Dataset を確認）")

# ---- vLLM ----
# wheelhouse Dataset（vllm 0.32.0 + 依存 wheel / 3.25 GB）を 2 つに分けて持つ:
#   vllm-wheelhouse        … 190 wheel（torch 2.13.0 + cu13 一式）
#   vllm-wheelhouse-patch  … nvidia-cublas 13.1.1.3 のみ
# `cuda-toolkit 13.0.3` が cublas を `==13.1.1.3.*` で固定するため、本体側の
# 13.8.2.12 だけでは `--no-index` が解決できない（CPU 実測 2026-10-31）。
# 解決に失敗したら PyPI にフォールバックする（起動できない方が損失が大きい）。
_WH_SLUGS = ("vllm-wheelhouse", "vllm-wheelhouse-patch")
_pip_extra = []


def _whl_dir(root):
    if _glob.glob(os.path.join(root, "*.whl")):
        return root
    for _s in sorted(_glob.glob(os.path.join(root, "*"))):
        if os.path.isdir(_s) and _glob.glob(os.path.join(_s, "*.whl")):
            return _s
    return None


def _unzip_wheels(root, tag):
    """--dir-mode zip の Dataset を 1 回だけ展開する。"""
    _zips = (_glob.glob(os.path.join(root, "*.zip"))
             + _glob.glob(os.path.join(root, "*", "*.zip")))
    if not _zips:
        return None
    import time as _t
    import zipfile as _zf
    _dst = os.path.join("/kaggle/tmp/whl", tag)
    os.makedirs(_dst, exist_ok=True)
    _t0 = _t.time()
    print(f"📦 {tag} の zip を展開: {[os.path.basename(z) for z in _zips]}")
    for _z in _zips:
        with _zf.ZipFile(_z) as _f:
            _f.extractall(_dst)
    print(f"   展開 {_t.time() - _t0:.1f}s")
    return _whl_dir(_dst) or _dst


_wh_dirs = []
for _slug in _WH_SLUGS:
    _root = kc_find_dataset(_slug)
    if not os.path.isdir(_root) or not os.listdir(_root):
        print(f"ℹ {_slug} なし")
        continue
    _d = _whl_dir(_root) or _unzip_wheels(_root, _slug)
    if not _d:
        print(f"⚠ {_slug} に wheel が無い")
        continue
    _n = len(_glob.glob(os.path.join(_d, "*.whl")))
    print(f"📦 ローカル wheelhouse: {_slug}（{_n} wheel）{_d}")
    _wh_dirs.append(_d)

if _wh_dirs:
    _pip_extra = ["--no-index"]
    for _d in _wh_dirs:
        _pip_extra += ["--find-links", _d]
else:
    print("ℹ wheelhouse Dataset なし → PyPI から取得（5〜10分）")
    # 「Dataset を付け忘れた」と「マウント先が想定と違う」を切り分ける。
    # これが無いと、metadata を確認するために 1 往復（15 分）を使うことになる。
    _seen = []
    for _pat in ("/kaggle/input/*", "/kaggle/input/datasets/*/*"):
        _seen += sorted(_p for _p in _glob.glob(_pat) if os.path.isdir(_p))
    for _p in _seen[:12]:
        print(f"    {_p}")

# ---- warm cache（任意 Dataset）----
# torch.compile の AOT 成果物と Triton JIT を Dataset から復元する。
# コンテナは毎回まっさらなので、これがあると compile 30〜60s を省ける。
# キャッシュキーが違えば単にミスするだけ（壊れない）。
#
# 復元の ON/OFF を外から切り替えられるようにしてある（`KC_WARM_CACHE=0`）。
# Dataset を付けたまま「復元しない」比較を playground で取りたいため。
# これが無いと、Dataset を付けた瞬間に A/B の対照群が消える。
#
# 3 値にしてある:
#   "0"       … 復元しない（対照群）
#   "1"       … あれば復元。無ければ「なし」と出して続行（本番の既定）
#   "require" … **無ければ失敗**。実験用
#
# `require` が要る理由: `1` のままだと Dataset を付け忘れた実験が
# **baseline とまったく同じ結果を出し**、「キャッシュは効かない」という
# 偽陰性になる。1 往復（約 40 分）を捨てるより、その場で落ちたほうが安い。
# 実測 2026-11-01: `vllm-warm-cache` はまだ Kaggle に存在せず、playground の
# push が `not valid dataset sources` で落としていた（＝気づかないまま走る）。
_KC_WC_MODE = os.environ.get("KC_WARM_CACHE", "1").strip().lower()
_USE_WARM_CACHE = _KC_WC_MODE not in ("0", "off", "no", "false")
_WC_REQUIRED = _KC_WC_MODE in ("require", "required")
# 復元の結果は `timings` に残す。**print だけだと 1 回の実行から読み取れない**
# （ログは流れるが、`cli timings` で横並びにできない）。
_WC_INFO = {"mode": _KC_WC_MODE, "restored": False, "source": "", "tar": "",
            "mib": 0.0, "entries": 0, "seconds": 0.0, "missing": []}
# 「完全な tar か」の判定に**件数を使わない**。件数は Triton の版で変わるし、
# しきい値を置くと「正当な tar を弾く」ほうに倒れる。代わりに**名前で見る**。
#
# 実測 2026-11-01（`vllm-playground` の 2 本の tar を数えた）:
#   不完全: 30 エントリ / 70 KB   … profile run に到達していない
#   完全  : 2,077 エントリ / 195 MB / カーネル名 140 種
# 差は GDN（gated delta net）の chunked カーネル群が入っているか。
# 373 秒の主役はこの 2 つ（名前は vLLM の fla 実装にハードコードされている）:
#   chunk_fwd_kernel_o                                 x252
#   chunk_gated_delta_rule_fwd_kernel_h_blockdim64      x84
# **`chunk_o` / `chunk_delta_h` で探してはいけない**。実際のファイル名は
# 上のように長く、部分一致しないので「完全な tar を不完全と誤判定」する
# （最初にこの 2 つで書いて、v4 の tar を弾きかけた）。
# 末尾の `_blockdim64` は head dim を名前に含めたもので、版が変わると
# 変わりうる。だから**前側だけ**を条件にする。
_WC_REQUIRED_KERNELS = ("chunk_fwd_kernel_o", "chunk_gated_delta_rule_fwd_kernel_h")
_wc = kc_find_dataset("vllm-warm-cache")
# `kc_find_dataset` は**見つからなくても既定パスを返す**（`/kaggle/input/<slug>`）。
# そのため `if _wc:` だけでは常に真になり、tar が 1 本も無いときに**何も
# 表示しないまま素通り**していた（実測 2026-11-01: ログに warm cache の行が
# 1 行も出ず、効いているのか判別できなかった）。
#
# **Kaggle は Dataset に上げた `.tar` を展開して保持する。** 実測 2026-11-01:
# 195 MB の tar を `datasets create` したら、マウント側は
# `root/.triton/cache/<hash>/...` の木になっていて `*.tar` は 1 本も無かった。
# 「tar が無い＝キャッシュが無い」と読むと**正当な Dataset を落とす**ので、
# tar と展開済みの木の両方を受ける。
def _kc_wc_tree(root):
    """Dataset の中の Triton キャッシュのディレクトリ（無ければ ""）。"""
    if not root or not os.path.isdir(root):
        return ""
    base = root.rstrip(os.sep).count(os.sep)
    for _dp, _dn, _fs in os.walk(root):
        if (os.path.basename(_dp) == "cache"
                and os.path.basename(os.path.dirname(_dp)) == ".triton"):
            return _dp
        if _dp.count(os.sep) - base >= 4:      # これ以上は潜らない
            _dn[:] = []
    return ""


_tars = [] if not _USE_WARM_CACHE else (
    _glob.glob(os.path.join(_wc, "*.tar"))
    + _glob.glob(os.path.join(_wc, "*", "*.tar")))
_tree = "" if not _USE_WARM_CACHE else _kc_wc_tree(_wc)
if not _USE_WARM_CACHE:
    print("ℹ warm cache: KC_WARM_CACHE=0 なので復元しない（A/B の対照）")
elif _tars or _tree:
    import shutil as _sh2
    import tarfile as _tf2
    # 復元先を**数える**。`extractall("/")` は tar の arcname 次第で
    # どこへでも落ちるので、サイズだけ出していると「復元したのに 1 件も
    # 当たらない」を 1 往復使って調べることになる。
    # Triton は 1 カーネル 1 ディレクトリなので直下の数で足りる。
    _tc = os.path.join(os.path.expanduser("~"), ".triton", "cache")
    _t0 = __import__("time").time()
    # tar があれば tar（展開済みの木は見ない＝二重にコピーしない）。
    _srcs = list(_tars) if _tars else [_tree]
    for _s in _srcs:
        try:
            if os.path.isdir(_s):
                _mib = 0.0
                for _r, _d, _fs in os.walk(_s):
                    for _fn in _fs:
                        try:
                            _mib += os.path.getsize(os.path.join(_r, _fn))
                        except OSError:
                            pass
                _mib = round(_mib / 1e6, 1)
                os.makedirs(_tc, exist_ok=True)
                _sh2.copytree(_s, _tc, dirs_exist_ok=True)
                _src = _s
            else:
                _mib = round(os.path.getsize(_s) / 1e6, 1)
                with _tf2.open(_s) as _f:
                    _f.extractall("/")
                _src = os.path.basename(_s)
            _n = len(os.listdir(_tc)) if os.path.isdir(_tc) else 0
            # 入っているカーネル名を集める。Triton は 1 カーネル 1 ディレクトリで、
            # 中に `<カーネル名>.json` / `.ptx` などを置く。名前で見るのは
            # 件数より意味がはっきりしているため（件数は Triton の版で変わる）。
            _names = []
            for _r, _d, _fs in os.walk(_tc):
                for _fn in _fs:
                    _names.append(_fn.lower())
            _missing = [k for k in _WC_REQUIRED_KERNELS
                        if not any(k in s for s in _names)]
            _WC_INFO.update(restored=True, source=_src, tar=_src, mib=_mib,
                            entries=_n, missing=_missing,
                            seconds=round(__import__("time").time() - _t0, 1))
            print(f"🧠 キャッシュ復元: {_src} {_mib:.0f} MB "
                  f"({_WC_INFO['seconds']}s) / Triton エントリ {_n}（{_tc}）"
                  + (f" / ⚠ 欠け: {', '.join(_missing)}" if _missing else ""))
        except Exception as _e:
            print(f"⚠ キャッシュ復元をスキップ: {_e}")
    # 復元はできたが**中身が薄い**場合。不完全な tar を復元すると一部だけ当たって
    # 「少し効いた」ように見え、373 秒が戻る理由を「キャッシュは効かない」と
    # 誤読する。`require` はその誤読を防ぐためのモードなので、ここも落とす。
    if _WC_REQUIRED and not _WC_INFO["restored"]:
        raise RuntimeError(
            f"KC_WARM_CACHE=require なのに復元できなかった: {_srcs}。"
            "抽出／コピーで例外が出ている（上の ⚠ を参照）。")
    if _WC_REQUIRED and _WC_INFO["missing"]:
        raise RuntimeError(
            f"KC_WARM_CACHE=require なのに "
            f"{', '.join(_WC_INFO['missing'])} が入っていない: "
            f"{_WC_INFO['source']} {_WC_INFO['mib']} MB / "
            f"エントリ {_WC_INFO['entries']}。"
            "この 2 つは 373 秒の主役（profile run の GDN Triton JIT）で、"
            "無いということは**起動に成功していない回**のキャッシュということ。"
            "不完全なまま測ると『キャッシュは効かない』という偽陰性になる。"
            "`KC_EXPERIMENT=baseline` の成果物から採り直すこと"
            "（判断できる場合は KC_WARM_CACHE=1 でこの検査を外せる）。")
elif _WC_REQUIRED:
    # ここで落とす。続行すると baseline と同じ数字が出て、効かなかったのか
    # Dataset が無かったのか区別できないログだけが残る。
    raise RuntimeError(
        f"KC_WARM_CACHE=require なのに tar も展開済みの木も無い: {_wc}"
        f"（exists={os.path.isdir(_wc)}）。"
        "Dataset `vllm-warm-cache` が未作成か、空のまま push している。"
        "このまま走らせると baseline と同じ結果になり、"
        "「キャッシュは効かない」という偽陰性になるので中断する。")
else:
    # 「Dataset を付け忘れ」と「中身が入っていない」を切り分ける。
    print(f"ℹ warm cache なし（compile をやり直す）: {_wc}"
          f" exists={os.path.isdir(_wc)}")

# ---- torch を「消してから入れる」ための道具 ----
# `pip uninstall torch` では足りない（実測 v7）。uninstall は RECORD に載っている
# ファイルしか消さないので、**新しい版で削除されたファイル**は残る。結果、
# 2.11 の `torch/utils/_config_module.py`（`deprecated=` を知らない Config）と
# 2.13 の `torch/_dynamo/config.py`（`Config(deprecated=True)` を呼ぶ）が同居し、
#   TypeError: Config() got an unexpected keyword argument 'deprecated'
# になる。torch 自身のファイル同士が食い違っているので、版を揃え直す以外に直せない。
def _site_dirs():
    r = subprocess.run(
        [sys.executable, "-c",
         "import sysconfig;print(sysconfig.get_paths()['purelib'])"],
        capture_output=True, text=True, timeout=120)
    d = (r.stdout or "").strip()
    return [d] if d and os.path.isdir(d) else []


def _purge_torch():
    """site-packages の torch / torchvision / torchaudio 系を物理削除する。

    `torch-*` はハイフン付きなので torchmetrics / torchcodec を巻き込まない
    （巻き込むと無関係なパッケージまで壊す）。
    """
    n = 0
    for _d in _site_dirs():
        for _pat in ("torch", "torch-*", "torchvision", "torchvision-*",
                     "torchaudio", "torchaudio-*", "torchgen", "torchgen-*",
                     "functorch", "functorch-*"):
            for _p in _glob.glob(os.path.join(_d, _pat)):
                try:
                    if os.path.isdir(_p):
                        shutil.rmtree(_p, ignore_errors=True)
                    else:
                        os.remove(_p)
                    n += 1
                except OSError as _e:
                    print(f"⚠ 削除できず: {_p} ({_e})")
    return n


def _torch_diag():
    """torch が混ざっているかを 1 画面で分かる形で出す。

    これが無いと「import に失敗」だけ見て 6 分を無駄にし、原因を推測する羽目になる。
    `_config_module.py` に `deprecated` が無ければ、そちらが旧版の残骸。
    """
    out = []
    for _d in _site_dirs():
        for _p in sorted(_glob.glob(os.path.join(_d, "torch*"))):
            out.append(f"  {os.path.basename(_p)}")
        _cm = os.path.join(_d, "torch", "utils", "_config_module.py")
        if os.path.isfile(_cm):
            _src = open(_cm, encoding="utf-8", errors="replace").read()
            out.append(f"  _config_module.py {len(_src)}B "
                       f"deprecated={'deprecated' in _src}")
        else:
            out.append("  _config_module.py なし")
    return "\n".join(out) or "  (site-packages が見つからない)"


def _probe(code, timeout=900):
    """別プロセスで import を試す。親で import すると半端な状態が尾を引く。"""
    r = subprocess.run([sys.executable, "-c", code],
                       capture_output=True, text=True, timeout=timeout)
    return r.returncode == 0, ((r.stdout or "") + (r.stderr or ""))[-1200:]


_r = subprocess.run([sys.executable, "-c", "import vllm"],
                    capture_output=True, text=True)
if _r.returncode == 0:
    print("✅ vLLM は既にインストール済み")
else:
    # ドライバと現行 torch を先に記録する。CUDA 13 系の wheel を入れると
    # `libcudart.so.13` 不在や「driver version is insufficient」で落ちるので、
    # 失敗したときに原因を 1 往復で特定できるようにしておく。
    _drv = subprocess.run(
        ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
        capture_output=True, text=True)
    _tv = subprocess.run(
        [sys.executable, "-c", "import torch;print(torch.__version__, torch.version.cuda)"],
        capture_output=True, text=True)
    print(f"🧩 driver: {(_drv.stdout or '').strip().splitlines()[:1]} / "
          f"torch: {(_tv.stdout or '').strip()}")
    print(f"⬇️  vLLM をインストール中: {VLLM_PIP_SPEC}（5〜10分）")
    # 依存も一緒に入れる（`--no-deps` を付けない）。
    # 理由: vLLM 0.32.0 は CUDA 13 ビルドで、イメージの torch 2.11.0+cu128 のままでは
    # `ImportError: libcudart.so.13` で起動できない（実測 2026-10-31）。torch 2.13.0 と
    # nvidia cu13 一式まで入れて初めて噛み合う。ドライバは 580.178.04 で CUDA 13 対応済み。
    #
    # ただし**上書きインストールは不可**。2.11 のファイルが残って 2.13 と混ざり、
    # `TypeError: Config() got an unexpected keyword argument 'deprecated'`
    # （torch._dynamo）のような内部不整合になる（実測）。しかも pip uninstall では
    # 消し切れない（新版で削除されたファイルが残る）ので、物理削除する。
    print("🧹 site-packages の torch 系を物理削除してから入れる")
    print(f"   {_purge_torch()} 個のパスを削除")

    # ---- インストーラの選択（実測で決めた）----
    # CPU ベンチ `vllm-install-bench`（2026-11-01、`pip install --target` で同じ
    # 190 wheel を 4 通りに展開）:
    #
    #   pip（暖）        200.1s   81,844 files
    #   pip --no-compile 112.6s   56,416 files
    #   uv               35.7s   56,812 files
    #
    # 差の大半は **pyc の生成（25,428 個）と pip の直列展開**。転送ではない
    # （wheelhouse を付けても pip は 199.6 → 230.4s と遅くなった）。
    #
    # ⚠ **この比は本番に移らなかった**（実測 2026-11-01 / v12）。
    #   bench は `--target` の**まっさらなディレクトリ**へ入れる。本番は
    #   torch を物理削除した site-packages へ入れる。同じ wheel 同じホストでも
    #   pip 261.6s → **uv 1151.7s（4.4 倍遅い）**。理由は未同定（uv のキャッシュ
    #   経路が GCSFUSE 上の wheel を引き直している疑い）。**既定は pip**。
    #   uv を試すときは `KC_INSTALLER=uv` を明示した回だけにする（既定を壊さない）。
    _want_uv = os.environ.get("KC_INSTALLER", "").strip().lower() == "uv"
    _uv = None
    if _want_uv:
        _uv = shutil.which("uv")
        if not _uv:
            _ru = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                                  "--no-input", "uv"],
                                 capture_output=True, text=True, timeout=600)
            _uv = shutil.which("uv")
            print(f"⬇️  uv を用意: {_uv}（rc={_ru.returncode}）")
        if not _uv:
            print("⚠ uv を用意できない → pip で入れる")
    _installer = "pip"
    _p = None
    if _uv:
        _uv_cmd = ([_uv, "pip", "install", "--python", sys.executable,
                    "--quiet"] + _pip_extra + [VLLM_PIP_SPEC])
        print(f"⬇️  vLLM を uv でインストール: {VLLM_PIP_SPEC}"
              f"（wheelhouse {len(_wh_dirs)} 個）")
        try:
            _p = subprocess.run(_uv_cmd, capture_output=True, text=True,
                                timeout=1800)
        except subprocess.TimeoutExpired:
            print("⚠ uv が 1800 秒で終わらない → pip に切り替え")
            _p = None
        if _p is not None:
            print(((_p.stdout or "") + (_p.stderr or ""))[-1500:])
            # uv は entry point を作るが、`--no-index` で解決が欠けたときに
            # **部分的に成功**して rc=0 を返すことがある。`vllm` CLI の実在まで
            # 確かめる（無いと serve が始まらない）。
            if _p.returncode == 0 and shutil.which("vllm"):
                _installer = "uv"
            else:
                print(f"⚠ uv が不完全（rc={_p.returncode}, "
                      f"vllm={shutil.which('vllm')}）→ pip で入れ直す")
    if _installer != "uv":
        _p = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-input"]
            + _pip_extra + [VLLM_PIP_SPEC],
            capture_output=True, text=True, timeout=3600,
        )
        print((_p.stdout or "")[-2000:])
        print((_p.stderr or "")[-2000:])
        if _p.returncode != 0 and _pip_extra:
            # wheelhouse の版ずれ等で解決できないことがある（実測: cublas 13.1.1.3）。
            # ここで止めると 1 回分の GPU を捨てるので、ネットワークに切り替えて再試行。
            print("⚠ ローカル wheelhouse で解決失敗 → PyPI にフォールバック")
            _pip_extra = []
            _p = subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", "--no-input",
                 VLLM_PIP_SPEC],
                capture_output=True, text=True, timeout=3600,
            )
            print((_p.stdout or "")[-2000:])
            print((_p.stderr or "")[-2000:])
        if _p.returncode != 0:
            fail("vLLM のインストールに失敗しました（上のログ参照）")
    print(f"   install 手段: {_installer}")
    _mark("pip")

    # 入れた直後に別プロセスで確かめる。ここで落ちるなら torch はまだ混ざっている。
    # torchaudio の調整に進んでも直らない種類の失敗なので、構成を出して**即座に**止める
    # （先へ進むと 5 分以上無駄にする）。
    _ok_imp, _imp_out = _probe(
        "import torch, vllm;print(torch.__version__, torch.version.cuda, "
        "torch.__file__);print('vllm', vllm.__version__)")
    if not _ok_imp:
        print("⚠ インストール直後の import に失敗。torch の構成を出す:")
        print(_torch_diag())
        print(_imp_out)
        fail("vLLM の import に失敗しました（torch の構成は上記）")

# torchaudio は transformers が import する。vLLM の pip install で torch だけが
# 差し替わると CUDA 版数が不一致になり、vllm CLI が即死する
# （実測: vllm 0.32.0 / torch 2.13.0+cu130 に対し torchaudio 2.8.3+cu128）。
#
# 検出は「実際に落ちる経路」で行う。`import vllm.entrypoints.cli.main` だけでは
# 不十分だった（v2 実測）: benchmark.main 等は main() の中で遅延 import される
# ため、深い依存（transformers → torchaudio）の不整合を見逃す。
# torchaudio の整合は serve より前に単独で確認する（v2 の真因）。
# PyPI の torchaudio 2.11.0 は +cu130 ビルド（wheel の version.py で実測）なので、
# torch 2.13.0+cu130 とそのまま一致する。
def _vllm_pin(name):
    """vLLM の METADATA が要求する版を取り出す（例: 'torchaudio==2.11.0'）。

    torchaudio の版は **torch の版ではない**。vllm 0.32.0 は torch==2.13.0 と
    同時に torchaudio==2.11.0 を要求する（METADATA 実測）。torch に合わせて
    2.13.0 を探すと存在せず、1 往復無駄にする。
    """
    r = subprocess.run(
        [sys.executable, "-c",
         "import importlib.metadata as m;print(chr(10).join(m.requires('vllm') or []))"],
        capture_output=True, text=True, timeout=120)
    for _ln in (r.stdout or "").splitlines():
        _ln = _ln.split(";")[0].strip()          # 環境マーカーは落とす
        if _ln.lower().startswith(name + "=="):
            return _ln
    return None


# 版が同じでも**ビルド**が違うと落ちる。pip は `2.11.0+cu128` を `==2.11.0` の
# 充足とみなす（PEP 440 ではローカル版を無視する）ので、イメージ残りの +cu128 が
# そのまま残り「PyTorch has CUDA 13.0 whereas TorchAudio has 12.8」になる（実測）。
# PyPI の torchaudio 2.11.0 は +cu130 ビルドなので、--force-reinstall で置き換える。
_TA_SPEC = _vllm_pin("torchaudio") or "torchaudio==2.11.0"


def _ta_check():
    r = subprocess.run([sys.executable, "-c", "import torchaudio"],
                       capture_output=True, text=True, timeout=300)
    return r.returncode == 0, ((r.stdout or "") + (r.stderr or ""))[-900:]


_ta_good, _ta_err = _ta_check()
if not _ta_good:
    print(f"⚠ torchaudio 不整合 → {_TA_SPEC} を --no-deps で入れ直す")
    print(_ta_err)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input"]
                   + _pip_extra
                   + ["--no-deps", "--force-reinstall", _TA_SPEC],
                   capture_output=True, text=True, timeout=1800)
    _ta_good, _ta_err = _ta_check()
if not _ta_good:
    # vLLM が torchaudio を使うのは音声モデルだけ（midashenglm/audio.py 等）。
    # このモデルは音声不要なので、外すのが最終手段として妥当。
    print("⚠ まだ不整合 → torchaudio を削除")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                    "torchaudio"], capture_output=True, text=True, timeout=600)
    _ta_good, _ta_err = _ta_check()
print(f"torchaudio import: {'OK' if _ta_good else 'NG'}")
_mark("torchaudio")

_PREFLIGHT_CMD = ["vllm", "serve", "--help"]


def _cli_ok():
    r = subprocess.run(_PREFLIGHT_CMD, capture_output=True, text=True,
                       timeout=300)
    out = (r.stdout or "") + (r.stderr or "")
    bad = ("Traceback" in out) or ("RuntimeError" in out) or len(out) < 500
    return (r.returncode == 0 and not bad), out[-1500:]


_ok, _err = _cli_ok()
if not _ok:
    # transformers 側は ImportError を握るので、外すのが最短（通信不要）。
    # vLLM 0.32 は torchaudio==2.11.0 を要求する（METADATA 実測）。
    # 版を揃えて入れ直すのが本筋。駄目なら外す（transformers は ImportError を握る）。
    print(f"⚠ vLLM CLI が起動しない → torchaudio を {_TA_SPEC} で入れ直す")
    print(_err)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input"]
                   + _pip_extra
                   + ["--no-deps", "--force-reinstall", _TA_SPEC],
                   capture_output=True, text=True, timeout=1800)
    _ok, _err = _cli_ok()
if not _ok:
    print("⚠ まだ失敗 → torchaudio を外して再試行")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                    "torchaudio"], capture_output=True, text=True, timeout=600)
    _ok, _err = _cli_ok()
if not _ok:
    print("⚠ まだ失敗 → torchaudio を torch に合わせて入れ直す")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input",
                    "-U", "torchaudio"], capture_output=True, text=True,
                   timeout=3600)
    _ok, _err = _cli_ok()
if not _ok:
    # help が壊れていても serve が壊れているとは限らない（実測: CPU では
    # transformers ではなく arg_utils 側で別の失敗をする）。ここで止めず、
    # 起動を試す。壊れていれば最初の試行が即座に落ちるだけで済む。
    print("⚠ vLLM CLI の確認に失敗 → そのまま起動を試す")
    print(_err)
_mark("cli_help")

try:
    import torch
    import vllm
except Exception as _e:
    # import 失敗は「torch の入れ替わり」で起きやすい。原因を残して止める。
    fail(f"vllm/torch の import に失敗: {type(_e).__name__}: {_e}")
print(f"✅ vLLM {vllm.__version__} / torch {torch.__version__} "
      f"/ CUDA {torch.version.cuda}")
print(f"   GPU 認識: {torch.cuda.device_count()} 枚")
_mark("import")

# 比較可能な形で残す。`install_seconds` はテンプレートの timings が読む。
# セルから data を直接触らない理由は kc_save_timings の説明を参照。
#
# `torch_after_install` を別名で残すのは、result.json の `torch` が
# **install 前**（イメージ同梱版）の値だから。実測 2026-11-01 は
# `torch: 2.11.0+cu128`（イメージ）→ `torch 2.13.0+cu130`（実際に動いた版）で、
# 表だけ見ると「2.11 で動いた」と誤読する。
kc_save_timings({"install_seconds": _MARKS["import"],
                 "install_marks": dict(_MARKS),
                 "installer": _installer,
                 "torch_after_install": torch.__version__,
                 "cuda_after_install": torch.version.cuda,
                 "wheelhouse": {"dirs": len(_wh_dirs),
                                "no_index": bool(_pip_extra)},
                 # 復元の有無と**エントリ数**。`entries` が小さいまま起動が
                 # 速くならなければ「効いていない」ではなく「tar が不完全」。
                 "warm_cache_restore": dict(_WC_INFO),
                 "vllm_version": vllm.__version__})
print(f"⏱ install 内訳: {_MARKS}")

# ---- プロキシ + UI ----
with open(PROXY_PATH, "wb") as f:
    f.write(base64.b64decode(PROXY_SRC_B64))
with open(CHAT_PATH, "wb") as f:
    f.write(base64.b64decode(CHAT_SRC_B64))
print(f"✅ auth_proxy.py: {os.path.getsize(PROXY_PATH):,} B")
print(f"✅ llama-chat.html: {os.path.getsize(CHAT_PATH):,} B")


---
### 🌐 リバースプロキシ起動（UI 配信 + 同一オリジン化）

推論サーバは非公開、公開側は認証プロキシが受けます。


In [ ]:
# リバースプロキシを起動（UI を / で配信し、API を同一オリジンにする）
# 起動・待機・認証なし検査の実装は共通ランタイム（runtime/proxy.py）。
# 中継先・追加引数・認証は settings セルが定義する（PROXY_* / UPSTREAM_PORT）。
import os

# 認証は既定で無効。PROXY_AUTH に API_PASS が入っているときだけ有効になる。
proxy_env = {**os.environ, **PROXY_AUTH}
proxy_proc = kc_start_proxy(
    PROXY_PORT, f"http://127.0.0.1:{UPSTREAM_PORT}", CHAT_PATH, PROXY_PATH,
    extra_args=PROXY_EXTRA_ARGS, env=proxy_env, fail=fail,
)

# 推論サーバ起動前なので 5xx は正常。401/403 が返るなら認証の設定ミス。
# 完全判定（200 / streaming / OpenAI 互換）は公開URL経由の verify セルへ一本化。
kc_assert_no_auth(PROXY_PORT, path=PROXY_CHECK_PATH, fail=fail)


---
### 🔥 vLLM サーバー起動（初回は 5〜10 分）

起動条件は **上から順に試行**し、失敗したら次へ落ちます。

| # | MTP | max-model-len | 備考 |
|---|---|---|---|
| 1 | ON | 32768 | 本命 |
| 2 | ON | 16384 | KV 不足時 |
| 3 | OFF | 16384 | MTP 非対応アーキのとき |
| 4 | OFF | 8192 | 最小構成 |
| 5 | OFF | 8192 | V0 エンジン（T4 用フォールバック） |

`--speculative-config` などのフラグは `vllm serve --help` を見て
**存在するものだけ**付けます（バージョン差で即死しないため）。

> 🎛 プロキシが `temperature<=0` を 0.7 に、`repetition_penalty` を 1.05 に
> 矯正します（`--enforce-sampling`）。temperature=0 で崩壊させないため。


In [ ]:
# vLLM 起動 → /v1/models が返るまで待機（失敗時は条件を落として再試行）
import json
import os
import re
import shutil
import signal
import subprocess
import threading
import time
import urllib.error
import urllib.request

STAGE = "/kaggle/tmp/vllm-model"
MTP_FILE = "model-mtp.safetensors"

# 起動ログの「節目」。末尾 1 行だけを見ていると、長い工程に入った瞬間から
# 何も分からなくなる。実測 2026-10-31: torch.compile 終了（514s）から CUDA graph
# capture 開始（874s）まで **360 秒**、ログは `shm_broadcast ... 60 seconds` の
# 繰り返しだけで、重みロードの 2 回目なのか autotune なのか特定できなかった。
# 節目を別枠で拾えば「最後に越えた関門」が常に分かる。
_MILESTONES = (
    "Starting to load model", "load_format", "Loading weights took",
    "Model loading took", "init engine", "Starting torch.compile",
    "torch.compile took", "available_kv_cache_memory", "Memory profiling",
    "Capturing CUDA graphs", "Graph capturing finished", "init distributed",
    "Starting vLLM API server", "route",
)

# ログ行頭の時刻（`10-31 13:22:08`）。日付は使わず時刻だけ取り出す。
_TS_RE = re.compile(r"\b\d{2}-\d{2} (\d{2}):(\d{2}):(\d{2})\b")


def _phase_marks(log_text, t0_wall):
    """起動ログから「各節目に到達した時刻（起動からの秒）」を取り出す。

    60 秒ポーリングでは graph capture（実測 41 秒）すら分解できない。
    vLLM のログ行には `10-31 13:22:08` 形式の時刻が入っているので、起動時刻と
    突き合わせれば **1 秒精度**になる。これが無いと「360 秒の無音区間」の
    正体を毎回推測することになる（実測 2026-10-31）。

    同じ節目が複数回出たら**最初**を採る。最後を採ると、再試行やサブプロセスの
    重複出力で区間が縮んで見える。取れなかった節目はキーごと欠落させる
    （0 で埋めると「一瞬で終わった」と誤読する）。
    """
    out = {}
    for ln in log_text.splitlines():
        m = _TS_RE.search(ln)
        if not m:
            continue
        t = int(m.group(1)) * 3600 + int(m.group(2)) * 60 + int(m.group(3))
        if t + 43200 < t0_wall:      # 0 時をまたいだ
            t += 86400
        for name in _MILESTONES:
            if name in ln and name not in out:
                out[name] = round(t - t0_wall, 1)
    return out


_TS_ONLY = re.compile(r"\b\d{2}-\d{2} \d{2}:\d{2}:\d{2}\b")


def _long_gaps(log_text, t0_wall, min_seconds=30, keep=5):
    """時刻付き行の間隔から「無音区間」を抽出する（長い順）。

    vLLM は**長い工程に入ると何も出さない**。実測 2026-11-01: torch.compile
    終了（282s）→ CUDA graph capture 開始（661s）の **379 秒**は
    `No available shared memory broadcast block ... 60 seconds` だけが並ぶ。
    区間の長さは分かっても中身が分からないので、**無音の直前と直後の行**を
    対で残す。次に疑うべき対象はこの 2 行で決まる。

    同じ警告が 60 秒おきに繰り返す場合は 1 本に**連結**する。連結しないと
    60 秒 × 5 本が並び、「381 秒詰まった」ではなく「60 秒が 5 回」に見える。
    判定は「区切りの行が**埋め草**（同じ本文が何度も出る行）かどうか」で行う。
    埋め草は無音の**中身**であって終わりではないので、次の無音と繋げる。
    """
    stamps = []
    for ln in log_text.splitlines():
        m = _TS_RE.search(ln)
        if not m:
            continue
        t = int(m.group(1)) * 3600 + int(m.group(2)) * 60 + int(m.group(3))
        if t + 43200 < t0_wall:      # 0 時をまたいだ
            t += 86400
        stamps.append((t - t0_wall, ln))

    def _body(s):
        """時刻を落とした本文（同じ行の繰り返しを判定するため）。"""
        return _TS_ONLY.sub("", s).strip()

    seen = {}
    for _, ln in stamps:
        b = _body(ln)
        seen[b] = seen.get(b, 0) + 1

    gaps = []
    for (ta, la), (tb, lb) in zip(stamps, stamps[1:]):
        if tb - ta < min_seconds:
            continue
        gaps.append({"from": round(ta, 1), "seconds": round(tb - ta, 1),
                     "before": la.strip()[:160], "after": lb.strip()[:160],
                     "_ab": _body(lb)})

    merged = []
    for g in gaps:
        prev = merged[-1] if merged else None
        # 直前の無音が**埋め草で終わっている**なら、無音はまだ続いている。
        if prev is not None and seen.get(prev["_ab"], 0) > 1:
            prev["seconds"] = round(g["from"] + g["seconds"] - prev["from"], 1)
            prev["after"] = g["after"]
            prev["_ab"] = g["_ab"]
        else:
            merged.append(g)
    for g in merged:
        g.pop("_ab", None)
    merged.sort(key=lambda g: -g["seconds"])
    return merged[:keep]


def _phase_spans(marks):
    """主要区間の所要（秒）。片方の節目が無ければ None（0 と区別する）。"""
    pairs = {
        "weights": ("Starting to load model", "Starting torch.compile"),
        "compile": ("Starting torch.compile", "torch.compile took"),
        "after_compile": ("torch.compile took", "Capturing CUDA graphs"),
        "capture": ("Capturing CUDA graphs", "Graph capturing finished"),
    }
    out = {}
    for name, (a, b) in pairs.items():
        if a in marks and b in marks and marks[b] >= marks[a]:
            out[name] = round(marks[b] - marks[a], 1)
        else:
            out[name] = None
    return out



def _stage_model(with_mtp):
    """モデルを symlink で 1 か所にまとめる（19 GB をコピーしない）。

    MTP の draft head (model-mtp.safetensors) は、speculative を切った状態で
    読ませると unexpected key になるため、その場合だけ除外する。
    """
    if os.path.isdir(STAGE):
        shutil.rmtree(STAGE)          # symlink のみなので実体は消えない
    os.makedirs(STAGE, exist_ok=True)
    n = 0
    for fn in sorted(os.listdir(MODEL_SRC)):
        if fn.startswith("."):
            continue
        if fn == MTP_FILE and not with_mtp:
            continue
        src = os.path.join(MODEL_SRC, fn)
        if not os.path.isfile(src):
            continue
        os.symlink(src, os.path.join(STAGE, fn))
        n += 1
    return n


DRAFT_DIR = "/kaggle/tmp/vllm-draft"
# draft に必要なのは mtp の重み + config 類だけ。残すと `*.safetensors` の glob が
# 本体 17.41 GiB まで拾い、draft が同じだけ読み直す（実測 §27）。
DRAFT_KEEP = ("config", "tokenizer", "chat_template", "generation_config",
              "special_tokens_map", "vocab", "merges", "added_tokens")


def _stage_draft():
    """draft (MTP) 専用の最小ディレクトリを作る。戻り値は symlink 本数。"""
    if not os.path.isfile(os.path.join(MODEL_SRC, MTP_FILE)):
        return 0
    if os.path.isdir(DRAFT_DIR):
        shutil.rmtree(DRAFT_DIR)
    os.makedirs(DRAFT_DIR, exist_ok=True)
    n = 0
    for fn in sorted(os.listdir(MODEL_SRC)):
        if fn.startswith("."):
            continue
        if fn != MTP_FILE and not fn.startswith(DRAFT_KEEP):
            continue
        src = os.path.join(MODEL_SRC, fn)
        if not os.path.isfile(src):
            continue
        os.symlink(src, os.path.join(DRAFT_DIR, fn))
        n += 1
    return n


# ---- 利用可能なフラグを確認（バージョン差の吸収）----
# vLLM 0.32 の `serve --help` は Config Group の一覧しか出さず、
# 個々のフラグは `--help=all` にしか無い（GPU 実測 2026-10-29）。
# 旧実装は 2,134 文字の要約を「完全な help」と誤認し、_has() が全フラグを
# 偽判定 → MTP / LMO / chat-template が全部落ちた。既知フラグの有無で
# 「完全な help か」を判定し、ダメなら VERIFIED_FLAGS に倒す。
def _run_help(arg):
    _p = subprocess.run(["vllm", "serve", arg], capture_output=True,
                        text=True, timeout=180)
    return _p.returncode, (_p.stdout or "") + (_p.stderr or "")


def _looks_complete(text):
    """個々のフラグまで含む help か。要約だけなら False。"""
    return (len(text) > 5000
            and "--tensor-parallel-size" in text
            and "--max-model-len" in text)


_rc, HELP = _run_help("--help=all")
if not _looks_complete(HELP):
    _rc2, _h2 = _run_help("--help")
    if len(_h2) > len(HELP):
        HELP, _rc = _h2, _rc2
# ヘルプ本文を成果物に残す（フラグ判定が外れたときの唯一の手掛かり）
save_output("vllm_help.txt", HELP[:20000])
print(f"  vllm serve --help=all: exit={_rc} {len(HELP)} 文字 "
      f"complete={_looks_complete(HELP)}")
print(HELP[:1200])
HELP_OK = (_rc == 0 and _looks_complete(HELP)
           and "Traceback (most recent call last)" not in HELP)
if not HELP_OK:
    print("⚠ 完全な help が取れない → VERIFIED_FLAGS で判定する")


# vllm 0.32.0 wheel の arg_utils/cli_args を静的に走査して実在を確認したもの。
# （詳細 notes/vllm-cpu-findings.md §22）
VERIFIED_FLAGS = {
    "--trust-remote-code", "--reasoning-parser", "--language-model-only",
    "--limit-mm-per-prompt", "--enable-prefix-caching", "--enforce-eager",
    "--tool-call-parser", "--enable-auto-tool-choice", "--speculative-config",
    "--spec-method", "--kv-cache-dtype", "--enable-log-requests",
    "--chat-template", "--chat-template-content-format", "--default-chat-template-kwargs",
    "--attention-backend", "--max-num-batched-tokens", "--generation-config",
    "--override-generation-config", "--load-format", "--tokenizer",
}


def _has(flag):
    """ヘルプ本文の「行頭オプション」として存在するか。

    単純な部分文字列一致だと説明文中の言及で誤判定するため行頭で見る。
    ヘルプが取れない環境では楽観側（True）に倒さず、wheel を静的に走査して
    実在を確認した VERIFIED_FLAGS だけを許可する。楽観側だと 0.32 で消えた
    フラグを付けて未知引数で即死する（過去に実際に踏んだ）。
    """
    if not HELP_OK:
        # 事前に wheel を静的解析して「実在する」と確認済みのものだけ許可する。
        # ここを楽観（True）にすると、0.32 で消えた --chat-template のような
        # フラグを付けて即死する（実測で該当）。
        return flag in VERIFIED_FLAGS
    return re.search(r"^\s*" + re.escape(flag) + r"\b", HELP,
                     re.MULTILINE) is not None


# 必須フラグ（ここが通らなければ何を試しても無駄）
CORE_ARGS = [
    "vllm", "serve", STAGE,
    "--served-model-name", SERVED_NAME,
    "--host", "127.0.0.1",
    "--port", str(VLLM_PORT),
    "--tensor-parallel-size", str(TP_SIZE),
    "--dtype", "half",
    # 量子化の指定は試行ごとに変える（下の PLAN の 7 要素目）。
    # このモデルの quantization_config は format="pack-quantized"（= compressed-tensors）
    # であり、awq 固定は誤り。auto（指定なし）を第一候補にする。
    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--gpu-memory-utilization", str(GPU_MEM_UTIL),
    "--kv-cache-dtype", KV_CACHE_DTYPE,
]

# 任意フラグ。バージョンによって引数値が違うことがあるので、
# 後半の試行では落として「フラグが原因の失敗」を切り分ける。
OPT_ARGS = []
if _has("--trust-remote-code"):
    OPT_ARGS.append("--trust-remote-code")
# 0.32 では --disable-log-requests が廃止され --enable-log-requests に反転。
# 省略しても害がないので、存在した場合だけ付ける（誤指定で試行を落とさない）。
if _has("--disable-log-requests"):
    OPT_ARGS.append("--disable-log-requests")
if _has("--reasoning-parser"):
    # Qwen3 系の思考タグを reasoning_content に分離する
    OPT_ARGS += ["--reasoning-parser", "qwen3"]
if _has("--enable-auto-tool-choice") and _has("--tool-call-parser"):
    OPT_ARGS += ["--enable-auto-tool-choice", "--tool-call-parser", "qwen3_xml"]

# ---- 試行計画（上から順に）----
# 各要素: (ラベル, speculative config, max-model-len, 追加env, 任意フラグを使うか,
#          追加引数, 量子化引数)
# 量子化は「指定なし（auto）」を第一候補。config.json の quantization_config は
# format="pack-quantized" で、AWQ 形式ではない（cfgdeep 実測）。
SPEC = None
if MTP_ENABLED and _has("--speculative-config"):
    SPEC = {"method": MTP_METHOD, "num_speculative_tokens": MTP_TOKENS}
else:
    print("ℹ --speculative-config 非対応 or MTP=0 → MTP なしで起動")

# draft を最小ディレクトリに向ける（17.41 GiB の二重読みを回避）。
# 失敗しても次の段が draft dir なしで再試行するので、ここは楽観で良い。
SPEC_DIR = None
if SPEC:
    _dn = _stage_draft()
    if _dn:
        SPEC_DIR = {**SPEC, "model": DRAFT_DIR}
        print(f"  draft dir : {DRAFT_DIR}（symlink {_dn} 個）")
    else:
        print("ℹ mtp ファイルが無い → draft は target と同じ dir を使う")

# 注意: 0.32 に V0 エンジンは無く、VLLM_USE_V1 / VLLM_ATTENTION_BACKEND は
# envs.py に定義が無い（= 未知 env として警告されるだけ。設定しても無効）。
# T4(sm75) では FlashAttention backend が capability>=8.0 を要求して弾かれ、
# 既定で TRITON_ATTN が選ばれる（triton_attn.supports_compute_capability=True）。
# よってフォールバック段は「明示的に TRITON_ATTN を指定する」に置き換える。

# 視覚塔を落としてテキスト専用で起動する（全段の既定）。
#  - テキスト配信しかしないので不要。0.86 GiB と mm profiling の VRAM を節約。
#  - 実装は StageMissingLayer で子モジュール登録されないため未使用キー警告も出ない
#    （models/utils.py:756 "Don't register this as a child module in order to
#      avoid missing keys when loading weights"）。
# なお quantization_config.ignore の `model.visual.*` は「prefix 文字列」と
# 完全一致で照合される（should_ignore_layer → is_equal_or_regex_match）。
# 視覚塔の prefix は `model.visual...` なので一致する（keymap の key 比較では
# 0 件だったが、判定に使うのは key ではなく prefix）。`re:.*mtp.*` も同様に
# draft の prefix `mtp.*` に一致する。
LMO = ["--language-model-only"] if _has("--language-model-only") else []
if not LMO:
    print("ℹ --language-model-only 非対応 → 視覚塔ありで起動")

AUTO_Q = []                                    # 指定なし（config から自動判定）
CT_Q = ["--quantization", "compressed-tensors"]  # 明示（pack-quantized）
# --quantization awq / auto は config の compressed-tensors と不一致で
# ModelConfig の検証に落ちる（cfgresolve 実測）。試行段からは外した。

STARTUP_BUDGET = int(os.environ.get("VLLM_STARTUP_BUDGET", "4200"))   # 70 分
ATTEMPT_TIMEOUT = int(os.environ.get("VLLM_ATTEMPT_TIMEOUT", "4200"))  # 1 試行に総予算を渡す
# 10/31 のリセット後（30h）前提の値。install は wheelhouse Dataset で
# 60〜120s になる見込み（未実測）。短い予算で回すときは環境変数で下げる。
# 注意: 残クォータが 30 分未満のときにこの設定で push しないこと。
#
# **なぜ 1800/900 をやめたか（2026-11-01 実測）**
# `baseline` が 2 試行とも「切られた」だけで、壊れてはいなかった:
#   attempt 1 … 900 秒（ATTEMPT_TIMEOUT）で終了、重みロードの途中
#   attempt 2 … 806 秒（残予算でクランプ）で SIGTERM、
#               08:07:53 開始 → 08:14:32 に embed_tokens まで到達、
#               その後 `Loading weights took` が出ないまま打ち切り
# 同じ回の vLLM ログは `Checkpoint size: 17.41 GiB / Available RAM: 5.85 GiB`
# で prefetch を切り、GCSFUSE からの読みが 20 MiB/s 前後まで落ちていた。
# 17.41 GiB ÷ 20 MiB/s ≒ 900 秒。**重みだけで 1 試行の枠を使い切る**ので、
# そのあとの profile run（373 秒）に絶対に届かない。
# 実測の幅は 103 MiB/s（out2g: 18.21 GiB / 181s）〜 20 MiB/s と 5 倍ある。
# 予算は**速い回**に合わせると遅い回が必ず落ちるので、遅い側に合わせる。
#
# **なぜ 2700 でも足りなかったか（2026-11-01 の v3 実測）**
# 2700 に上げた回は「切られる」のではなく、**最後の層まで届いてから**切られた:
#   t=250s   attempt 1 開始（install は 222s で完了していた）
#   t=2890s  layers.61 の最後のテンソル（= 全 62 層ぶんの読み込みが終わった）
#   t=2722s  ← ここで予算超過を判定し、残り 8 段を打ち切り
# つまり**重みだけで約 2,640 秒**。15.5 GiB を TP=2 の 2 ワーカーが
# それぞれ読むので実 I/O は約 31 GB、実効 11.8 MiB/s。まだ profile run
# （373 秒）が残るので、2,640 + 373 ≒ 3,013 秒が下限。2,700 では 300 秒足りない。
# ホスト RAM は `avail=25,071 free=437 cache=23,455 MiB`、ワーカー RSS が
# 12.1 GB × 2。空きが 500 MiB を割ったまま読み続けており、ページキャッシュが
# 追い出されて GCSFUSE を読み直している疑いがある（→ notes §20）。
# 4,200 は 3,013 秒に対して約 1,180 秒の余裕。ノードが 1.4 倍遅くても届く。
#
# `ATTEMPT_TIMEOUT` を総予算と同値にしてあるのは、「遅い」失敗と
# 「非対応」失敗を区別するため。非対応は即座に落ちる（SGLang の sm80 など）ので
# 予算をほとんど食わず、次の段に回せる。遅いだけの試行を 15 分で切ると
# **必ず落ちる**うえ、次段も同じ理由で落ちる。

# 重みロードの prefetch（`notes/vllm-startup-speedup.md` §22）。
# vLLM 0.32 の `_prefetch_all_checkpoints` は `sorted_files[rank::world_size]`
# で**rank ごとにファイルを分担**して page cache に載せる。TP=2 でも総読みが
# チェックポイント 1 回分になる計算（いまは 2 ワーカーが各自 mmap して 2 回読む）。
# ただし既定のスレッド数 8 は CPU 実測で一番遅い形（8 本 2.3 MiB/s / 1 本 110 MiB/s）。
# **env が空なら 1 バイトも足さない**（既存の生成物と挙動を変えない）。
_PF_STRATEGY = os.environ.get("VLLM_SAFETENSORS_STRATEGY", "").strip()
_PF_THREADS = os.environ.get("VLLM_SAFETENSORS_PREFETCH_THREADS", "").strip()
LOAD_ARGS = []
if _PF_STRATEGY:
    if _has("--safetensors-load-strategy"):
        LOAD_ARGS += ["--safetensors-load-strategy", _PF_STRATEGY]
        # スレッド数は prefetch のときだけ意味がある（lazy/eager では無視される）
        if _PF_THREADS and _has("--safetensors-prefetch-num-threads"):
            LOAD_ARGS += ["--safetensors-prefetch-num-threads", _PF_THREADS]
    else:
        # 黙って落とすと「prefetch は効かなかった」という偽陰性になる
        print("⚠ --safetensors-load-strategy 非対応 → 重み prefetch は付けない")

# vLLM 側が prefetch を**辞退した**ときの文言（`weight_utils.py` の warning）。
# チェックポイント 18.21 GiB に対して RAM 32 GB は余裕が無く、同じコードでも
# 「載せる」ノードと「辞退する」ノードがある（実測 2026-11-02: 片方 1,081 秒 /
# 片方 47 秒per層）。辞退した回を同じ表に混ぜると施策の効果を読み違えるので、
# 見つけたら記録して警告を出す。**辞退しても起動は続く**（遅い経路に落ちるだけ）。
_PF_GUARD = "exceeds 90% of available RAM"
_PF_GUARD_HIT = ""

PLAN = []
# 予算 1500s で 1 試行 600s。MTP を 2 段続けると失敗時に予算を食い潰すので、
# 「MTP+draft dir」→ 失敗したら MTP を一旦あきらめて素の起動を優先する。
if SPEC_DIR:
    PLAN.append(("mtp+dd", SPEC_DIR, MAXLEN_CANDIDATES[0], {}, True, LMO, AUTO_Q))
# ここからは任意フラグを落として最小構成で試す
PLAN.append(("plain", None, MAXLEN_CANDIDATES[0], {}, False, LMO, AUTO_Q))
PLAN.append(("plain", None, MAXLEN_CANDIDATES[1], {}, False, LMO, AUTO_Q))
PLAN.append(("plain", None, 16384, {}, False, LMO, AUTO_Q))
# 2026-10-29 GPU 実測: 32k は torch.compile(61s) 完了後に
# "No available shared memory broadcast block found" を 60s 間隔で出し続け、
# 420s で打ち切った（= ハング or CUDA graph capture/ダミー実行が異常に遅い）。
# eager は graph capture を丸ごと飛ばすので、仮説の切り分けを最優先で入れる。
PLAN.insert(1, ("eager", None, MAXLEN_CANDIDATES[0], {},
                False, LMO + ["--enforce-eager"], AUTO_Q))
# 量子化の明示指定（auto が外した場合）
PLAN.append(("ct", None, 16384, {}, False, LMO, CT_Q))
# attention backend を明示（auto でも TRITON_ATTN になるが、選択を固定する）
PLAN.append(("triton", None, 8192, {}, False,
             LMO + ["--attention-backend", "TRITON_ATTN"], AUTO_Q))
# （eager は上の insert で 2 番目に移した。重複させない）
# MTP の再挑戦（draft dir 無し = 17 GiB を読み直すので最後に置く）
if SPEC:
    PLAN.append(("mtp", SPEC, MAXLEN_CANDIDATES[2], {}, True, LMO, AUTO_Q))
# language-model-only が原因で落ちた場合の退避。視覚塔は積むが modality を 0 に
# して実体化させない（get_limit_per_prompt=0 → 同じく StageMissingLayer 経路）。
PLAN.append(("vl0", None, 8192, {}, False,
             ["--limit-mm-per-prompt", '{"image":0,"video":0}'], AUTO_Q))
assert all(len(p) == 7 for p in PLAN), "PLAN の要素数が 7 でない"

# prefetch は「読み込み段」の話なので、全試行に同じフラグを足す。
# 段ごとに変えると、効いたときにどの段のおかげか分からなくなる（v14 の轍）。
if LOAD_ARGS:
    PLAN = [(_l, _s, _m, _e, _o, _f + LOAD_ARGS, _q)
            for (_l, _s, _m, _e, _o, _f, _q) in PLAN]
    print("ℹ 重み prefetch: " + " ".join(LOAD_ARGS))

# 予算が短いとき（< 900s）は「最も軽い仮説検証」を先頭に持ってくる。
# 実測: 27B の重みロードだけで約 250s、torch.compile が 61s。
# 64K+MTP+グラフを試す余裕は無いので、eager（graph capture 無し）で
# 「そもそも起動できるのか」を先に確定させる。
if STARTUP_BUDGET < 900:
    PLAN.sort(key=lambda _p: 0 if _p[0] == "eager" else 1)
    print("ℹ 起動予算が短い → eager を先頭に並べ替え")


def _gpu_used_mib():
    """全 GPU の使用 VRAM 合計（MiB）。取れなければ -1。"""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=20).stdout
        return sum(int(x) for x in re.findall(r"[0-9]+", out))
    except Exception:
        return -1


def _gpu_pids():
    """GPU を掴んでいる PID をドライバに聞く（`ps` より確実）。

    vLLM は `setproctitle` でプロセス名を `VLLM::Worker_TP0` のように書き換える。
    名前で殺しに行くと**綴りを外した瞬間に静かに失敗**する（実測 2026-11-01:
    `VLLM::WorkerTP` は `VLLM::Worker_TP0` に一致せず、18,194 MiB を掴んだまま
    5 試行を無駄にした）。ドライバに PID を聞けば名前は関係ない。
    """
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-compute-apps=pid",
             "--format=csv,noheader"],
            capture_output=True, text=True, timeout=20).stdout
        return [int(x) for x in re.findall(r"[0-9]+", out)]
    except Exception:
        return []


def _free_vram(limit_mib=2000, tries=18):
    """VRAM が実際に空くまで落とし続け、最後の使用量（MiB）を返す。

    「殺したつもり」で次の試行へ進むと、失敗理由が必ず
    `Free memory on device cuda:0 ... is less than desired GPU memory utilization`
    になり、原因（残骸）ではなく**症状**を追いかけることになる。
    """
    for i in range(tries):
        _u = _gpu_used_mib()
        if 0 <= _u < limit_mib:
            return _u
        if i in (0, 3, 8):        # 毎回ではなく段階的に掃除する
            for _pat in ("VLLM::Worker", "VLLM::EngineCore", "vllm serve",
                         "vllm.entrypoints"):
                subprocess.run(["pkill", "-9", "-f", _pat],
                               capture_output=True)
            for _pid in _gpu_pids():
                if _pid in (os.getpid(), os.getppid()):
                    continue      # 自分自身は殺さない
                try:
                    os.kill(_pid, signal.SIGKILL)
                    print(f"  🔫 GPU を掴んでいる PID {_pid} を停止")
                except OSError:
                    pass
        time.sleep(5)
    return _gpu_used_mib()


def _mem_info():
    """`/proc/meminfo` の主要値（MiB）。**空きだけ見ても原因は分からない。**

    実測 2026-11-01（v12）: 試行開始の 4 分前に `MemAvailable` が 30,315 MiB
    あったのに、vLLM が重みを読む時点では **5.68 GiB** まで落ちていた。
    その 24 GiB が何なのか（page cache か、anon か、/dev/shm か）を切り分けるには
    `Cached` / `Shmem` / `MemFree` を並べて見る必要がある。片方だけでは
    「誰かが食った」のか「キャッシュが張り付いた」のか決められない。
    """
    want = ("MemTotal", "MemFree", "MemAvailable", "Cached", "Shmem")
    out = {}
    try:
        for ln in open("/proc/meminfo", encoding="utf-8"):
            k = ln.split(":", 1)[0]
            if k in want:
                out[k] = int(ln.split()[1]) // 1024
    except (OSError, ValueError, IndexError):
        pass
    return out


def _mem_available_mib():
    """空き RAM（MiB）。重みロードが固まるかの予兆になる。"""
    return _mem_info().get("MemAvailable", -1)


def _cgroup_mib():
    """cgroup v2 の上限と現在使用量（MiB）。読めなければ `(None, None)`。

    vLLM の prefetch 判定は `min(psutil.available, 上限 - 使用量)` で、
    **使用量に page cache が含まれる**（`vllm/utils/cpu_resource_utils.py` の
    `get_available_memory`）。ここを `MemAvailable` で代用すると、同じ瞬間に
    「30 GiB 空いている」と「6 GiB しか無い」が両立して食い違う。

    実測 v17（2026-11-02）: こちら `MemAvailable` 30,159 MiB / vLLM 6.16 GiB。
    上限 32,098 MiB − 使用量 約 25,800 MiB ≒ 6.3 GiB で説明が付く。
    """
    out = []
    for path in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"):
        v = None
        try:
            with open(path, encoding="utf-8") as f:
                raw = f.read().strip()
            v = None if raw in ("", "max") else int(raw) // 2**20
        except (OSError, ValueError):
            v = None
        out.append(v)
    return out[0], out[1]


def _cgroup_unreclaimable_mib():
    """cgroup v2 の**追い出せない**メモリ（MiB）。読めなければ `None`。

    `memory.current` は page cache を含むので、cache が張り付いたノードでは
    「空き」を過小評価する（v17: 30,159 対 6,298 MiB）。`file` は追い出せるが
    `anon` / `shmem` / `kernel` は追い出せない。

    `kernel` が無いカーネル（5.15 未満）では**代用しない**。`slab` だけでは
    `kernel_stack` / `sock` / `pagetables` が抜けて過大評価になるので、
    `None` を返して呼び出し側を従来式（`memory.current`）に戻す。
    """
    want = ("anon", "shmem", "kernel")
    got = {}
    try:
        with open("/sys/fs/cgroup/memory.stat", encoding="utf-8") as f:
            for ln in f:
                parts = ln.split()
                if len(parts) == 2 and parts[0] in want:
                    got[parts[0]] = int(parts[1])
    except (OSError, ValueError):
        return None
    if "anon" not in got or "kernel" not in got:
        return None
    # 合算は **bytes のまま**。MiB に落としてから足すと切り捨てが積み上がる。
    return (got["anon"] + got.get("shmem", 0) + got["kernel"]) // 2**20


def _prefetch_avail_mib(exclude_cache=False):
    """vLLM が prefetch 可否に使う「空き」（MiB）。

    `exclude_cache=False` は vLLM 0.32 の式そのもの
    `min(MemAvailable, 上限 - memory.current)`。同じ瞬間に「30 GiB 空いている」
    と「6 GiB しか無い」が両立する（v17）。

    `exclude_cache=True` は `KC_PREFETCH_PATCH` で vLLM 側を差し替えたあとの式。
    page cache を除いた使用量（`anon` / `shmem` / `kernel`）で数える。
    内訳が読めないときは**従来式に戻す**（過大評価しない）。
    **表示と実際の判定を一致させる**ために、こちらも同じ式で出す。
    """
    avail = _mem_available_mib()
    limit, used = _cgroup_mib()
    if limit is None or used is None:
        return avail
    if exclude_cache:
        used = _cgroup_unreclaimable_mib() or used
    return max(0, min(avail, limit - used))


_PF_PATCH_MARK = "KC_PREFETCH_PATCH"

# vLLM の `get_available_memory` を置き換える本体。**ファイルへ追記**するのが
# 要点で、親で monkeypatch しても TP worker（別プロセスで `vllm` を import し
# 直す）には届かない。追記なら全プロセスが同じ関数を読む。
_SITE_PATCH = '''# --- KC_PREFETCH_PATCH ---
# vLLM の get_available_memory は cgroup の memory.current を使う。この値には
# **page cache が含まれる**ため、前のジョブが張った cache で「RAM が足りない」と
# 誤判定する（実測 v17: MemAvailable 30,159 MiB を 6.16 GiB と判定）。
# page cache は追い出せるので、追い出せない分だけで数え直す。
# 内訳が読めない環境では**元の式に戻す**（過大評価して OOM にしない）。
def _kc_int(path):
    try:
        with open(path, encoding="utf-8") as _f:
            _raw = _f.read().strip()
        return None if _raw in ("", "max") else int(_raw)
    except (OSError, ValueError):
        return None


def _kc_stat():
    _out = {}
    try:
        with open("/sys/fs/cgroup/memory.stat", encoding="utf-8") as _f:
            for _ln in _f:
                _p = _ln.split()
                if len(_p) == 2:
                    _out[_p[0]] = int(_p[1])
    except (OSError, ValueError):
        return None
    return _out or None


def get_available_memory():
    import psutil
    _avail = psutil.virtual_memory().available
    _limit = _kc_int("/sys/fs/cgroup/memory.max")
    _stat = _kc_stat()
    if _limit is None:
        return _avail
    if _stat is None or "anon" not in _stat or "kernel" not in _stat:
        # 追い出せない分を正確に出せない → 元の式（memory.current）に任せる
        _cur = _kc_int("/sys/fs/cgroup/memory.current")
        return _avail if _cur is None else max(0, min(_avail, _limit - _cur))
    _used = _stat["anon"] + _stat.get("shmem", 0) + _stat["kernel"]
    return max(0, min(_avail, _limit - _used))
'''


def _vllm_cpu_utils_paths():
    """site-packages の `vllm/utils/cpu_resource_utils.py` を探す（import しない）。

    `import vllm` は重く、CUDA 初期化まで走り得る。パスだけ見れば足りる。
    """
    import site

    roots = list(sys.path)
    for getter in ("getsitepackages", "getusersitepackages"):
        try:
            got = getattr(site, getter)()
        except Exception:                                   # noqa: BLE001
            continue
        roots += [got] if isinstance(got, str) else list(got)
    seen, out = set(), []
    for root in roots:
        if not root or root in seen:
            continue
        seen.add(root)
        p = os.path.join(root, "vllm", "utils", "cpu_resource_utils.py")
        if os.path.isfile(p):
            out.append(p)
    return out


def _patch_vllm_avail_memory():
    """vLLM の空き判定を page cache 除外版に差し替える（ファイル追記）。

    `pip install` の**後**に呼ぶ。戻り値は timings に残す辞書。

    追記は**一時ファイル + `os.replace`**（部分書き込みを残さない）。同じ VM で
    再実行しても marker で二重追記しない。1 つでも失敗したら `failed` に入れ、
    呼び出し側は差し替えを**有効扱いにしない**（ゲートだけ緩むのを防ぐ）。
    """
    paths = _vllm_cpu_utils_paths()
    if not paths:
        return {"patched": False, "reason": "cpu_resource_utils.py が見つからない"}
    done, failed = [], []
    for path in paths:
        try:
            with open(path, encoding="utf-8") as f:
                src = f.read()
            mode = os.stat(path).st_mode
        except OSError as e:
            failed.append(f"{path}: {type(e).__name__}: {e}")
            continue
        if _PF_PATCH_MARK not in src:
            tmp = path + ".kc-patch.tmp"
            try:
                with open(tmp, "w", encoding="utf-8") as f:
                    f.write(src + "\n\n" + _SITE_PATCH)
                os.chmod(tmp, mode)
                os.replace(tmp, path)
            except OSError as e:
                try:
                    os.remove(tmp)
                except OSError:
                    pass
                failed.append(f"{path}: {type(e).__name__}: {e}")
                continue
            # 念のため .pyc を捨てる（mtime だけで判定されると古い本体が使われる）
            cache = os.path.join(os.path.dirname(path), "__pycache__")
            try:
                for fn in os.listdir(cache):
                    if fn.startswith("cpu_resource_utils."):
                        os.remove(os.path.join(cache, fn))
            except OSError:
                pass
        done.append(path)
    return {"patched": bool(done) and not failed, "paths": done, "failed": failed}


def _mem_line():
    """1 行にまとめた RAM の内訳。無音区間の原因を後から追うための控え。"""
    m = _mem_info()
    if not m:
        return "RAM: 取得できません"
    limit, used = _cgroup_mib()
    cg = f" / cgroup {used}/{limit}" if limit is not None and used is not None else ""
    return (f"RAM 空き {m.get('MemAvailable', -1)} / 空 {m.get('MemFree', -1)}"
            f" / cache {m.get('Cached', -1)} / shm {m.get('Shmem', -1)} MiB"
            f"（全 {m.get('MemTotal', -1)}{cg}）")


def _top_rss(n=3):
    """RSS 上位 n 件（`名前:MiB`）。24 GiB を誰が持っているかを名前で見る。"""
    rows = []
    try:
        for pid in os.listdir("/proc"):
            if not pid.isdigit():
                continue
            try:
                with open(f"/proc/{pid}/statm", encoding="utf-8") as f:
                    rss = int(f.read().split()[1]) * 4096 // (1024 * 1024)
                with open(f"/proc/{pid}/comm", encoding="utf-8") as f:
                    name = f.read().strip()
            except (OSError, ValueError, IndexError):
                continue
            if rss:
                rows.append((rss, f"{name}:{rss}"))
    except OSError:
        return ""
    rows.sort(reverse=True)
    return " ".join(t for _, t in rows[:n])


# 署名から潰す「揺れる値」。同じ失敗なのに毎回違う文字列が入ると、
# ラダーの打ち切り判定が効かない。
_SIG_NOISE = (
    # 先頭に `\b` を付けない: `psm_3e24306e` の `3` の前は `_`（単語文字）なので
    # 単語境界にならず、付けると**ハッシュが消えない**（実測 2026-11-01）。
    (re.compile(r"[0-9a-f]{6,}"), "<hex>"),            # psm_3e24306e / dist_...
    (re.compile(r"pid=\d+"), "pid=<n>"),
    (re.compile(r"\b\d{2}-\d{2} \d{2}:\d{2}:\d{2}\b"), "<ts>"),
    (re.compile(r"\b\d+\.\d+\b"), "<f>"),
)


def _sig_of(tail, n=3):
    """失敗の署名。**揺れる値を潰してから**比べる。

    実測 2026-11-01: VRAM 未解放という**同一の理由**で 5 回失敗したのに、
    末尾に毎回違う `{'/psm_3e24306e'}` のハッシュが入るため署名が一致せず、
    ラダーが 5 回続行して約 900 秒を捨てた。原因ではなく**偶然**を比べていた。
    """
    lines = [ln.strip() for ln in tail.splitlines() if ln.strip()][-n:]
    s = " | ".join(lines)
    for _pat, _rep in _SIG_NOISE:
        s = _pat.sub(_rep, s)
    return s[:400]


def _kill(proc):
    """vLLM のプロセス木ごと落とし、VRAM が実際に空くまで待つ。

    2026-10-29 実測: 試行 1 を terminate した直後に試行 2 を起動したところ、
    125 秒で "Engine core initialization failed. See root cause above."
    で即死した（EngineCore / Worker が残って VRAM を掴んでいた）。
    SIGTERM は API サーバーにしか届かず、孫（Worker_TP*）は残る。
    """
    try:
        _pgid = os.getpgid(proc.pid)
        os.killpg(_pgid, signal.SIGTERM)
        proc.wait(timeout=20)
        os.killpg(_pgid, signal.SIGKILL)   # 残った子を確実に落とす
    except Exception:
        pass
    # /dev/shm の残骸を掃除する。SharedMemory が unlink されずに残ると
    # 次の試行の check_shm_free_space で Engine core が即死する
    # （実測: attempt 2/3 が 125 秒で "Failed core proc(s): {}"、
    #   同時に resource_tracker が /psm_* のリークを警告）。
    _left = 0
    try:
        for _fn in os.listdir("/dev/shm"):
            if _fn.startswith(("vllm", "psm_")):
                try:
                    os.unlink(os.path.join("/dev/shm", _fn))
                    _left += 1
                except Exception:
                    pass
    except Exception:
        pass
    if _left:
        print(f"  🧽 /dev/shm の残骸 {_left} 件を削除")
    # 解放待ち。ここを飛ばすと次の試行の失敗理由が「前のプロセスの残骸」になり、
    # 切り分けが壊れる。**戻り値（実際に空いたか）を呼び出し側で判定する**。
    _u = _free_vram()
    print(f"  🧹 停止後の VRAM 使用量: {_u} MiB")
    return _u


def _models(timeout=8):
    """起動済みなら /v1/models の JSON を返す（未起動なら None）。"""
    try:
        with urllib.request.urlopen(
            f"http://127.0.0.1:{VLLM_PORT}/v1/models", timeout=timeout
        ) as r:
            return json.loads(r.read())
    except Exception:
        return None


def _smoke(attempts=2, timeout=90):
    """起動後に 1 回だけ生成させ、初回 forward が通るか確かめる。

    /v1/models が返るだけでは不十分。量子化カーネル（Marlin/Exllama）や
    GDN の Triton カーネルは **推論して初めて** 落ちることがあり、その場合
    サーバーは 500 を返し続ける。ここで落とせば次の段に進める。
    """
    body = json.dumps({
        "model": SERVED_NAME,
        "messages": [{"role": "user", "content": "Reply with the single word OK."}],
        "max_tokens": 16,
        "temperature": 0.7,
        "stream": False,
    }).encode()
    last = ""
    for i in range(attempts):
        req = urllib.request.Request(
            f"http://127.0.0.1:{VLLM_PORT}/v1/chat/completions",
            data=body, headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                d = json.loads(r.read())
            ch = (d.get("choices") or [{}])[0]
            msg = ch.get("message") or {}
            txt = (msg.get("content") or "") + (msg.get("reasoning_content") or "")
            return True, (f"tok={(d.get('usage') or {}).get('completion_tokens')} "
                          f"fin={ch.get('finish_reason')} "
                          f"text={txt[:40]!r}")
        except urllib.error.HTTPError as exc:
            detail = exc.read()[:300].decode("utf-8", "replace")
            last = f"HTTP {exc.code}: {detail}"
        except Exception as exc:
            last = f"{type(exc).__name__}: {exc}"
        print(f"    smoke {i+1}/{attempts} NG: {last[:200]}")
        time.sleep(3)
    return False, last


proc = None
argv_used = None
started = False
# GPU クォータは有限。ラダーを全部試すと 1 時間超になり、途中で打ち切られると
# 成果物が残らない。総予算を切って、必ずログを残して終わるようにする。
# クォータ残 39 分（2,395s）で pip install に 5 分前後かかる前提の値。
# 25 分使うと公開できたとしても数分で打ち切られるため短くしてある。
# 2026-10-29 実測: 27B ハイブリッド（GDN/Mamba）を T4 x2 で起動すると、
# torch.compile 61s の後に "No available shared memory broadcast block" が
# 60s 間隔で出続け、420s では足りなかった（ハングか graph capture が異常に遅い）。
# **クォータ 30h のリセット後は 1 試行 15 分で回す**。
# 10/28 の残 39 分では 720/420 に落として使った（その値では足りなかった）。
_attempts = []
_prev_sig = None
_t_start = time.time()
tail = ""          # 予算切れで 1 度も走らなかった場合に備えて初期化
last_log = ""
smoke_fail = ""    # 起動はしたが生成で落ちた場合の HTTP 応答（ログに出ない）
save_output("vllm_attempts.txt", "起動試行ログ（追記保存）\n")

# 空き RAM は「重みロードが固まるか」の予兆になる。実測 2026-11-01:
# `Available RAM: 6.12 GiB` / checkpoint 18.21 GiB のとき、GCSFUSE 上の
# safetensors を mmap で読む工程が **840 秒間 1 行も進まなかった**
# （ログ最終行は `Loading safetensors checkpoint shards: 50%` のまま）。
# ここで気づけないと「起動が遅い」ではなく「原因不明のハング」に見える。
_mem = _mem_available_mib()
print(f"🧠 {_mem_line()}")
print(f"   上位 RSS: {_top_rss(5)}")
if 0 <= _mem < 8192:
    print(f"  ⚠ 空き RAM が少ない（{_mem} MiB < 8192）。重みロードで詰まる可能性。"
          f" 症状は「ログの最終行から無進捗のまま試行上限まで待つ」")


def _weights_gib():
    """重み（*.safetensors）の合計 GiB。prefetch の可否を決める分母になる。"""
    total = 0
    try:
        for fn in os.listdir(MODEL_SRC):
            if fn.endswith(".safetensors"):
                total += os.path.getsize(os.path.join(MODEL_SRC, fn))
    except OSError:
        pass
    return round(total / 2**30, 2)


def _drop_caches():
    """clean page cache を捨てる。**prefetch を復活させるためだけ**にやる。

    vLLM は「checkpoint が空き RAM の 90% を超える」と auto-prefetch を切り、
    GCSFUSE 上の safetensors を頁単位で mmap する遅い経路に落ちる。v10 は
    同じ手順で 133 秒だったので、差は起動時点の `MemAvailable` だけ。
    `Cached` が空きを圧迫しているなら、ここで捨てれば条件を満たし直せる。

    **ただし Kaggle では書けない**（実測 2026-11-01 / `vllm-io-bench` v2）:
    `OSError: [Errno 30] Read-only file system: '/proc/sys/vm/drop_caches'`。
    それでも呼ぶのは、環境が変わったときに 1 行で気づけるようにするため。
    結果（OK/NG）は必ず表示する。
    """
    before = _mem_info()
    ok = False
    try:
        with open("/proc/sys/vm/drop_caches", "w", encoding="utf-8") as f:
            f.write("1\n")          # 1 = page cache のみ（dentry/inode は残す）
        ok = True
    except OSError as e:
        return False, f"{type(e).__name__}: {e}"
    after = _mem_info()
    return ok, (f"{before.get('MemAvailable', -1)} → "
                f"{after.get('MemAvailable', -1)} MiB"
                f"（cache {before.get('Cached', -1)} → {after.get('Cached', -1)}）")


_WEIGHTS_GIB = _weights_gib()
# vLLM のしきい値（90%）に少し余裕を足す。ここを割ると prefetch が切られる。
_NEED_MIB = int(_WEIGHTS_GIB * 1024 / 0.9 * 1.05) if _WEIGHTS_GIB else 0

# 辞退の**原因**（page cache を空きに数えない）を消す。既定で入れる。
# `KC_PREFETCH_PATCH=0` で従来の挙動に戻せる（A/B の対照群）。
_PF_PATCH = None
if os.environ.get("KC_PREFETCH_PATCH", "1").strip().lower() not in (
        "0", "off", "false", "no"):
    _PF_PATCH = _patch_vllm_avail_memory()
else:
    _PF_PATCH = {"patched": False, "reason": "KC_PREFETCH_PATCH=0"}
_PF_PATCH_OK = (bool(_PF_PATCH.get("patched"))
                and not _PF_PATCH.get("failed")
                # 内訳が読めない環境では差し替えても従来式に戻るので、
                # 「差し替え済み」と表示しない（ゲートだけ緩むのを防ぐ）。
                and _cgroup_unreclaimable_mib() is not None)

_PF_AVAIL_RAW = _prefetch_avail_mib()
_PF_AVAIL = _prefetch_avail_mib(exclude_cache=_PF_PATCH_OK)
# 空きが読めない（-1）ときは「辞退される」と決めつけない。判定できないだけ。
_PF_OK = (not _NEED_MIB) or _PF_AVAIL < 0 or _PF_AVAIL >= _NEED_MIB
print(f"🧮 重み {_WEIGHTS_GIB} GiB / prefetch に必要な空き {_NEED_MIB} MiB")
if _PF_PATCH_OK:
    print(f"🩹 vLLM の空き判定を差し替え（page cache を除外）: "
          f"{', '.join(_PF_PATCH.get('paths') or [])}")
    print(f"🧮 差し替え後の空き {_PF_AVAIL} MiB（元の式なら {_PF_AVAIL_RAW} MiB）"
          f" → prefetch {'通る' if _PF_OK else '辞退される見込み'}")
else:
    _why = _PF_PATCH.get("reason")
    if not _why and _PF_PATCH.get("failed"):
        _why = "; ".join(_PF_PATCH["failed"])
    if not _why and _PF_PATCH.get("patched"):
        _why = "memory.stat の内訳が読めない（従来式に戻る）"
    print(f"🩹 差し替えなし（{_why or '不明'}）")
if not _NEED_MIB or _PF_AVAIL < 0:
    print(f"🧮 vLLM の式での空き {_PF_AVAIL} MiB → 判定できない")
else:
    print(f"🧮 vLLM の式での空き {_PF_AVAIL} MiB → prefetch "
          + ("通る" if _PF_OK else "辞退される見込み（重みロードが 5 倍遅い経路）"))
kc_save_timings({"prefetch_patch": {**_PF_PATCH, "avail_before_mib": _PF_AVAIL_RAW,
                                    "avail_after_mib": _PF_AVAIL,
                                    "need_mib": _NEED_MIB}})


def _read_range(path, off, nbytes, threads=1, chunk_mib=32):
    """`off` から `nbytes` を `threads` 本の `pread` で読む（先読みの実体）。

    同じ fd を複数スレッドで `pread` するのは安全（読み取り専用・位置指定）。
    戻り値は読めたバイト数（**足りない分は黙って捨てない**。呼び側が比べる）。
    """
    chunk = chunk_mib * 1024 * 1024
    total = 0
    fd = os.open(path, os.O_RDONLY)
    try:
        def _one(o):
            n = min(chunk, off + nbytes - o)
            if n <= 0:
                return 0
            try:
                return len(os.pread(fd, n, o))
            except OSError:
                return 0

        with _pool(max(1, threads)) as ex:
            for got in ex.map(_one, range(off, off + nbytes, chunk)):
                total += got
    finally:
        os.close(fd)
    return total


def _prewarm_weights(threads=1, chunk_mib=32, budget_s=300, batch_mib=512):
    """重みを先に読んで page cache に載せる（**時間予算つき**）。

    実測 2026-11-01（`vllm-io-bench` v2・CPU・GPU 0 秒）:

    | 読み方 | 実測 |
    |---|---|
    | GCSFUSE 1 本（64 MiB バッファ） | **110〜130 MiB/s** |
    | GCSFUSE 2/4/8/16 本（32 MiB チャンク） | **2.2〜2.4 MiB/s** ← 13〜50 倍遅い |
    | page cache に載った後の mmap | **5,677 MiB/s** |

    **並列にすると壊滅的に遅い。** 1 GiB を 8 本で読むと 449 秒（1 本なら 33 秒）。
    v10 の重み 133 秒（140 MiB/s）を「並列の証拠」と読んだのは**誤り**で、
    あれは単に回線が速い回だっただけ（同じ 1 本読みでも 43.8 MiB/s の回がある）。

    → **既定は 1 本**。並列は `KC_PREWARM_THREADS` で明示したときだけ試し、
    必ず 1 本と実測で比べて速い方を採る（決め打ちにしない）。

    `budget_s` を超える見込みなら**始めない**（部分的な先読みは「残りが遅い」
    という症状を防げない）。走り出したあとで速度が崩れた場合も、予算で打ち切る。
    """
    files = []
    try:
        for fn in sorted(os.listdir(MODEL_SRC)):
            if fn.endswith(".safetensors"):
                files.append(os.path.join(MODEL_SRC, fn))
    except OSError:
        return None
    if not files:
        return None

    t0 = time.time()
    big = max(files, key=os.path.getsize)
    big_size = os.path.getsize(big)
    probe_n = min(64 * 2**20, big_size)
    probe_off = max(0, big_size - probe_n)      # 末尾から（本読みと窓をずらす）
    cands = [1]
    if int(threads) > 1:
        cands.append(max(1, int(threads)))
    probe = {}
    for n in cands:
        _t = time.time()
        got = _read_range(big, probe_off, probe_n, threads=n, chunk_mib=chunk_mib)
        _dt = time.time() - _t
        probe[n] = round(got / 2**20 / _dt, 1) if _dt else None
    best = max(probe, key=lambda n: probe[n] or 0)
    rate = probe[best] or 0
    total_bytes = sum(os.path.getsize(p) for p in files)
    projected = (total_bytes / 2**20 / rate) if rate else None

    # 予算に収まらないと分かっているなら**始めない**。部分的な先読みは
    # 「残りが遅い」という症状を防げないので、時間だけ払って何も変わらない。
    # 1.5 倍の余裕は、probe が 64 MiB と短く、本読みの実効速度とずれるため。
    if projected is not None and projected > budget_s * 1.5:
        return {"skipped": "over budget", "probe": probe, "threads": best,
                "mib_s": rate, "projected_s": round(projected, 1),
                "seconds": round(time.time() - t0, 1)}

    total = 0
    aborted = False
    for path in files:
        size = os.path.getsize(path)
        off = 0
        while off < size:
            if time.time() - t0 > budget_s:
                aborted = True
                break
            n = min(batch_mib * 2**20, size - off)
            total += _read_range(path, off, n, threads=best, chunk_mib=chunk_mib)
            off += n
        if aborted:
            break
    dt = time.time() - t0
    return {"bytes": total, "seconds": round(dt, 1),
            "mib_s": round(total / 2**20 / dt, 1) if dt else None,
            "threads": best, "probe": probe, "aborted": aborted,
            "budget_s": budget_s, "projected_s": projected}


def _pool(n):
    """スレッドプール。**I/O 待ちなので GIL は関係ない**（`os.pread` は解放する）。"""
    from concurrent.futures import ThreadPoolExecutor
    return ThreadPoolExecutor(max_workers=n)


def _ram_trace(stop, path, every=5.0):
    """起動中の RAM を 5 秒ごとに残す。**24 GiB が消える瞬間**を捕まえるため。

    実測 2026-11-01（v12）: 試行開始の 4 分前に `MemAvailable` 30,315 MiB、
    vLLM が重みを読む時点で 5.68 GiB。差分 24 GiB の正体が分からないまま
    対策を積むのは順序が逆。5 秒粒度なら「どのプロセスが、いつ、どれだけ」が
    そのまま読める。
    """
    t0 = time.time()
    try:
        with open(path, "a", encoding="utf-8") as f:
            while not stop.is_set():
                m = _mem_info()
                f.write(f"{time.time() - t0:7.1f}s avail={m.get('MemAvailable', -1)}"
                        f" free={m.get('MemFree', -1)} cache={m.get('Cached', -1)}"
                        f" shm={m.get('Shmem', -1)} top={_top_rss(3)}\n")
                f.flush()
                stop.wait(every)
    except OSError:
        pass


# ---- Triton キャッシュ（起動の 54% を占める無音の正体）----
# v14 実測（2026-11-01, T4, eager）の時系列:
#   03:52:25  Using LBNHC KV cache layout            ← 82s 地点
#   ……373 秒の無音（shm_broadcast が 60 秒ごとに 6 回）……
#   03:58:38  Available KV cache memory: 3.45 GiB
#   03:58:38  Warming up Qwen GDN Triton kernels     ← ここは 3 秒
#   03:58:41  init engine (profile, create kv cache, warmup model) took 376.24s
# つまり **名前の付いた warmup は 3 秒**で、コストはその手前
# （`determine_available_memory` の profile run）にある。ram_trace を見ると
# その間ワーカーの RSS が 6,987 → 8,361 MiB と 11 MiB/s で増え続けており、
# ハングではなく**コンパイルしている**。成果物は `~/.triton/cache` に落ちる。
#
# したがって「前回の `~/.triton` を持ち込む」が筋。ただし **キーが違えば黙って
# ミスする**（Triton のキーは Triton の版・対象 sm・ソースハッシュを含む）ので、
# 判定は「ファイルがある」ではなく **「エントリが増えなかった」**にする。
# 増えた数が残っていれば、373 秒が戻ったときに「ミスした」のか「別の原因」かを
# 区別できる（この区別が無いまま対策を積むと、v14 の先読みと同じことになる）。
#
# パスは `TRITON_CACHE_DIR` で**明示する**。既定は Triton の版によって
# `~/.triton/cache` と `~/.cache/triton` のどちらにもなり得て、復元側（01 の
# tar）と食い違うと「復元したのに 1 件も当たらない」になる。
_TRITON_CACHE = os.environ.get("TRITON_CACHE_DIR") or os.path.expanduser(
    "~/.triton/cache")


def _uniq_dirs(paths):
    """実体で重複を落とす。`~/.triton` が別名で 2 回入ると tar が二重になる。"""
    out, seen = [], set()
    for p in paths:
        try:
            real = os.path.realpath(p)
        except OSError:
            real = p
        if real not in seen:
            seen.add(real)
            out.append(p)
    return tuple(out)


_TRITON_DIRS = _uniq_dirs((
    os.path.dirname(_TRITON_CACHE.rstrip("/")) or _TRITON_CACHE,
    os.path.expanduser("~/.triton"),
    "/root/.triton",
    "/root/.cache/triton",
))


def _triton_snapshot():
    """Triton キャッシュの「いまの顔ぶれ」を返す（比較用）。

    名前の**集合**を返すのは、再利用されたかを判定するため。「ファイルがある」
    だけでは、前回と別のカーネルをコンパイルした（＝当たっていない）場合を
    区別できない。1 カーネル 1 ディレクトリなので直下の名前で足りる。
    """
    names, where = set(), ""
    cands = (_TRITON_CACHE,) + tuple(os.path.join(x, "cache")
                                     for x in _TRITON_DIRS)
    for d in cands:
        if not os.path.isdir(d):
            continue
        where = where or d
        try:
            names.update(e.name for e in os.scandir(d))
        except OSError:
            pass
    return {"dir": where, "names": names}


def _triton_report(before, after):
    """比較結果を **JSON に書ける形**にする（集合は落とす。timings が壊れる）。"""
    n_before, n_after = len(before["names"]), len(after["names"])
    new = len(after["names"] - before["names"])
    return {"dir": after["dir"] or before["dir"],
            "entries_before": n_before,
            "entries_after": n_after,
            "new_entries": new,
            # 今回使ったエントリのうち、前回からあった割合＝当たった割合。
            "reused_pct": (round(100.0 * (n_after - new) / n_after, 1)
                           if n_after else None)}


def _capture_warm_cache(limit_bytes=3e9):
    """`~/.triton` を tar にして成果物へ置く（次回の Dataset 用）。

    既存の `engine_cleanup()` も同じ tar を作るが、あれはキープアライブの
    **終了時**に走るため、v14 では一度も実行されなかった（ログにキャッシュの
    行が 1 本も無く、`warm-cache.tar` も成果物に無い）。起動が通った時点で
    キャッシュは揃っているので、ここで取る方が確実。

    3 GB を超えたら諦める（転送と容量の無駄）。**失敗しても本処理は止めない**
    — 止めると「キャッシュが取れなかっただけで起動できた回」を失う。
    """
    if os.environ.get("KC_CAPTURE_CACHE", "1") == "0":
        return {"skipped": "KC_CAPTURE_CACHE=0"}
    try:
        import tarfile
        dirs = [d for d in _TRITON_DIRS if os.path.isdir(d)]
        if not dirs:
            return {"skipped": "no triton dir"}
        size = 0
        for d in dirs:
            for root_, _dirs, files in os.walk(d):
                for f in files:
                    try:
                        size += os.path.getsize(os.path.join(root_, f))
                    except OSError:
                        pass
                if size > limit_bytes:
                    break
            if size > limit_bytes:
                break
        if not size:
            return {"skipped": "empty", "dirs": dirs}
        if size > limit_bytes:
            return {"skipped": "too big", "bytes": size, "dirs": dirs}
        out = os.path.join(OUT_DIRS[0], "warm-cache.tar")
        os.makedirs(OUT_DIRS[0], exist_ok=True)
        t0 = time.time()
        with tarfile.open(out, "w") as tf:
            for d in dirs:
                # 復元側（01 の `extractall("/")`）に合わせて `/` からの相対で入れる。
                tf.add(d, arcname=os.path.relpath(d, "/"))
        return {"path": out, "bytes": os.path.getsize(out),
                "source_bytes": size, "seconds": round(time.time() - t0, 1),
                "dirs": dirs}
    except Exception as e:                      # noqa: BLE001
        print(f"⚠ キャッシュ退避をスキップ: {type(e).__name__}: {e}")
        return {"error": f"{type(e).__name__}: {e}"}


# ---- 先読み（既定 OFF。2026-11-01 に既定を反転した）----
# **先読みは「152.5 秒の純コスト」では済まない。後続の重みロードまで 4.7 倍遅くする。**
#
# 実測の並び（すべてログの時刻差）:
#
#   out2g（10-31, 先読みなし）: 重みロード 約 181s → 起動 642.4s → **31.7 t/s**
#   v14 （11-01, 先読みあり）: 先読み 152.5s → 試行 1 の重みロード **705.00s**
#                              → 921s で打ち切り → eager に落ちて 4.5 t/s
#
# 同じマシン・同じ重み（18.21 GiB）で 181s と 705s。**先読み自身が測った速度は
# 122.2 MiB/s** なので、素直に読めば 18.21 GiB ÷ 122 MiB/s ≈ 150s。181s はこれと
# 一致し、705s（26 MiB/s）は**機械自身の実力の 4.7 倍遅い**。つまり読み出し中に
# 何かが競合している。
#
# 機構（推定・ただし数字は整合する）: 先読みが page cache を 25 GiB まで満たす →
# ワーカーが 8.88 GiB × 2 を anonymous で確保 → カーネルが page cache を 17.8 GiB
# 追い出す（＝先読みした重みが消える）→ 試行は cold read に戻り、しかも reclaim と
# 競合する。v14 の `ram_trace.txt` は `Cached` が 20〜23 GiB のまま推移しており、
# この押し出しと矛盾しない。
#
# したがって `KC_PREWARM=1` の明示が原則。ただし **prefetch が辞退される見込みの
# 回だけは自動で試す**（`KC_PREWARM` 未設定時）。辞退されると vLLM は GCSFUSE 上の
# safetensors をテンソル単位で読む経路に落ち、実測 v17 は 18.21 GiB に **6,128 秒**
# （3.2 MB/s）かかった。同じノードでも 1 本の逐次読みなら桁が違う（110〜130 MiB/s の
# 実測あり）ので、probe で速さを測ってから決める。遅いノードでは probe だけで
# `over budget` になり、先読みは**始まらない**（`_prewarm_weights` が判定する）。
# `KC_PREWARM=0` を明示すれば自動も止まる。
_PREWARM = None
_pw_mode = os.environ.get("KC_PREWARM", "").strip().lower()
_pw_auto = (_pw_mode in ("", "auto")) and not _PF_OK
_pw_want = _pw_mode not in ("", "auto", "0") or _pw_auto
if _WEIGHTS_GIB and _pw_want:
    # 既定は **1 本**。並列は GCSFUSE で 13〜50 倍遅い（実測 2026-11-01）。
    _pw_threads = int(os.environ.get("KC_PREWARM_THREADS", "1"))
    _pw_budget = int(os.environ.get("KC_PREWARM_BUDGET", "300"))
    print(f"🔥 重みを先に読む（{_WEIGHTS_GIB} GiB / {_pw_threads} 本 / "
          f"予算 {_pw_budget}s"
          f"{' / prefetch 辞退の見込みで自動' if _pw_auto else ''}）…")
    _PREWARM = _prewarm_weights(threads=_pw_threads, budget_s=_pw_budget)
    if _PREWARM and _PREWARM.get("skipped"):
        print(f"   ⏭ 先読みしない（{_PREWARM['skipped']} / "
              f"{_PREWARM.get('mib_s')} MiB/s / 見込み "
              f"{_PREWARM.get('projected_s')}s / probe {_PREWARM['probe']}）")
    elif _PREWARM:
        print(f"   {_PREWARM['mib_s']} MiB/s / {_PREWARM['seconds']}s"
              f" / {_PREWARM['threads']} 本 / probe {_PREWARM['probe']}"
              f"{' / ⚠ 予算切れ（部分）' if _PREWARM['aborted'] else ''}")
    if _PREWARM:
        _PREWARM["decided_by"] = "auto（prefetch が辞退される見込み）" if _pw_auto else "KC_PREWARM"
    kc_save_timings({"prewarm": _PREWARM})

# キャッシュの「起動前の顔ぶれ」を控える。試行後に差分を取れば
# **当たったかどうか**が分かる（`new_entries` が多い＝作り直した＝ミス）。
_TRITON_BEFORE = _triton_snapshot()
print(f"🧠 Triton キャッシュ: {len(_TRITON_BEFORE['names'])} エントリ"
      f"（{_TRITON_BEFORE['dir'] or 'なし'} / 環境変数 {_TRITON_CACHE}）")
kc_save_timings({"triton_cache_before": {
    "entries": len(_TRITON_BEFORE["names"]), "dir": _TRITON_BEFORE["dir"]}})

# 先読みは「起動の準備」であって起動試行ではない。予算（STARTUP_BUDGET）は
# 試行に使う。ここを混ぜると、先読みが 400 秒かかった回に試行の時間が
# 残らず「先読みしたせいで起動できなかった」という最悪の形になる。
_t_start = time.time()

# 起動中の RAM を 5 秒ごとに記録する（別スレッド）。ポーリングは 60 秒なので、
# そのままだと「30 GiB → 5.7 GiB」が 1 区間に潰れて誰も特定できない。
RAM_TRACE = "/kaggle/working/ram_trace.txt"
_RAM_STOP = threading.Event()
_RAM_THREAD = threading.Thread(target=_ram_trace, args=(_RAM_STOP, RAM_TRACE),
                               daemon=True)
_RAM_THREAD.start()


for idx, (label, spec, maxlen, env_extra, use_opt, extra_args,
          quant_args) in enumerate(PLAN, 1):
    _elapsed = time.time() - _t_start
    if _elapsed > STARTUP_BUDGET:
        print(f"⏱ 起動予算 {STARTUP_BUDGET}s 超過（{int(_elapsed)}s）"
              f" → 残り {len(PLAN) - idx + 1} 試行を打ち切り")
        break
    n_staged = _stage_model(with_mtp=spec is not None)
    cmd = CORE_ARGS + (OPT_ARGS if use_opt else [])
    cmd += ["--max-model-len", str(maxlen)]
    if spec:
        cmd += ["--speculative-config", json.dumps(spec)]
    cmd += extra_args
    cmd += quant_args
    # tokenizer_config.json に chat_template が無いので明示指定する。
    # 0.32 でも --chat-template は FrontendArgs のフィールドとして実在
    # （entrypoints/launchers/cli_args.py:86。dataclass 経由で自動登録されるため
    #  add_argument の grep では見つからない）。存在しなければ付けない。
    _tpl = os.path.join(STAGE, "chat_template.jinja")
    if os.path.isfile(_tpl) and _has("--chat-template"):
        cmd += ["--chat-template", _tpl]
    log_path = f"/tmp/vllm_{idx}.log"
    last_log = log_path
    # torch.compile / Triton JIT は CPU バウンド。vLLM は既定で 1 スレッドに
    # 落とす（"Reducing Torch threads from 2 to 1 for serving"）ため明示する。
    # `TRITON_CACHE_DIR` も明示する: Triton の既定は版によって
    # `~/.triton/cache` と `~/.cache/triton` のどちらにもなり得て、復元した
    # ディレクトリと食い違うと**1 件も当たらないまま 373 秒が戻る**。
    env_run = {**os.environ, "OMP_NUM_THREADS": os.environ.get("OMP_NUM_THREADS", "4"),
               "TRITON_CACHE_DIR": _TRITON_CACHE,
               **env_extra}
    print()
    print(f"--- 試行 {idx}/{len(PLAN)}: spec={label} "
          f"max_len={maxlen} env={env_extra or 'default'} "
          f"quant={quant_args[1] if quant_args else 'auto'} "
          f"opt_flags={'on' if use_opt else 'off'} "
          f"staged={n_staged} files ---")
    # 起動の直前に空きを確保し直す。ここを飛ばすと「prefetch 無効の遅い経路」に
    # 入り、重みロードが 800 秒進まないまま試行上限を焼く（実測 2026-11-01）。
    # 判定は **vLLM と同じ式**（`_prefetch_avail_mib`）。`MemAvailable` だけで
    # 見ると「足りている」と言ったまま実際には辞退される（実測 v17）。
    # 差し替え済みなら vLLM も page cache を除いた式で見るので、こちらも合わせる。
    _avail = _prefetch_avail_mib(exclude_cache=_PF_PATCH_OK)
    if _NEED_MIB and 0 <= _avail < _NEED_MIB:
        _limit, _used = _cgroup_mib()
        _formula = ("page cache を除外" if _PF_PATCH_OK
                    else f"cgroup {_used}/{_limit}")
        print(f"  🧠 prefetch 用の空き {_avail} MiB < 必要 {_NEED_MIB} MiB"
              f"（min(MemAvailable {_mem_available_mib()},"
              f" {_formula})）→ 辞退される")
        if _PF_PATCH_OK:
            print("  🧹 drop_caches は試さない（差し替え済みで cache を除外している）")
        else:
            _ok, _detail = _drop_caches()
            print(f"  🧹 drop_caches: {'OK' if _ok else 'NG'} {_detail}")
            _avail = _prefetch_avail_mib()
        if 0 <= _avail < _NEED_MIB:
            _rate = (_PREWARM or {}).get("mib_s") or 0
            _eta = (f"実測 {_rate} MiB/s なら重みだけで "
                    f"{int(_WEIGHTS_GIB * 1024 / _rate)} 秒"
                    if _rate else "先読みの実測が無いので ETA は出せない")
            print(f"  ⚠ それでも足りない（{_avail} MiB）。prefetch 無しの遅い経路"
                  f"に入る（{_eta}）。予算 {STARTUP_BUDGET}s に収まるか確認")
    with open(log_path, "w") as log:
        _pre = _gpu_used_mib()
        if _pre > 3000:
            print(f"  ⚠ 起動前の VRAM 使用量 {_pre} MiB（残骸の疑い）→ 掃除")
            # 掃除した**結果**を見る。ここを見ないと、解放できていないまま
            # 5 試行を「Free memory ... is less than desired GPU memory
            # utilization」で焼き捨てる（実測 2026-11-01: 18,194 MiB が
            # 6 試行ぶん残り、約 900 秒を捨てた）。
            _pre = _free_vram()
            print(f"  🧹 掃除後: {_pre} MiB")
            if _pre > 3000:
                fail(f"VRAM が解放されません（{_pre} MiB 使用中）。前の実行の"
                     f"残骸か他プロセスが掴んでいます。ここで止めます"
                     f"（起動しても 0.9 × VRAM を確保できず即死する）。",
                     last_log or log_path)
        # start_new_session で独立したプロセスグループにする（_kill で木ごと落とす）
        proc = subprocess.Popen(cmd, stdout=log, stderr=log, env=env_run,
                                start_new_session=True)
    print(f"🔄 vLLM PID: {proc.pid}  log: {log_path}")

    t0 = time.time()
    # ログの時刻（時:分:秒）と突き合わせるための起動時刻（秒・日内）。
    _lt = time.localtime()
    _t0_wall = _lt.tm_hour * 3600 + _lt.tm_min * 60 + _lt.tm_sec
    # 1 試行の上限は残予算でクランプする（最後の 1 段が予算超過で
    # 中途半端に切られ、ログが残らないのを避ける）。
    _tmo = min(ATTEMPT_TIMEOUT,
               max(45, int(STARTUP_BUDGET - (t0 - _t_start))))
    _t_print = 0.0
    _sz_prev = 0
    _poll_marks = {}
    _mark_last = "(節目なし)"
    while time.time() - t0 < _tmo:
        d = _models(timeout=3)
        if d and d.get("data"):
            started = True
            break
        if proc.poll() is not None:
            break
        _now = time.time() - t0
        if _now - _t_print >= 60:
            _t_print = _now
            # 「起動待ち Ns (0 models)」だけでは何で詰まっているか分からない。
            # ログ末尾をそのまま出す（GPU 実行を 1 回無駄にしないため）。
            try:
                _sz = os.path.getsize(log_path)
                _lns = [ln.strip() for ln in
                        open(log_path, errors="replace").read().splitlines()
                        if ln.strip()]
            except Exception:
                _sz, _lns = 0, []
            _last_line = _lns[-1][:170] if _lns else "(ログ空)"
            # 末尾 1 行は警告で埋まりやすい。最後に越えた節目を別に出す。
            _hit, _mark = None, "(節目なし)"
            for _ln in reversed(_lns):
                _h = next((m for m in _MILESTONES if m in _ln), None)
                if _h:
                    _hit, _mark = _h, _ln[:150]
                    break
            if _hit and _hit not in _poll_marks:
                # 60 秒分解能の控え。tqdm の行（graph capture）は**ログに時刻が
                # 無い**ので、ログ時刻だけでは拾えない。ポーリング側で補完する。
                _poll_marks[_hit] = round(_now, 1)
            if _hit:
                _mark_last = _mark
            # prefetch を辞退されたかを見る。ここを見ないと「prefetch を付けたのに
            # 遅い」を延々と読み違える（実測 2026-11-02）。ログ全体を毎回走査して
            # いるので追加コストはほぼ無い。
            if LOAD_ARGS and not _PF_GUARD_HIT:
                for _ln in _lns:
                    if _PF_GUARD in _ln:
                        _PF_GUARD_HIT = _ln.strip()[:300]
                        print("  ⚠ vLLM が prefetch を辞退した（RAM 判定）: "
                              + _PF_GUARD_HIT)
                        break
            _grow = _sz - _sz_prev
            _sz_prev = _sz
            _gpu = subprocess.run(
                ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True).stdout.strip().replace("\n", " ")
            print(f"  ⏳ 起動待ち {int(_now)}s"
                  f"（{len(d.get('data', [])) if d else 0} models）: "
                  f"{_last_line}")
            print(f"     log {_sz // 1024} KiB (+{_grow}) / 節目: {_mark}")
            print(f"     GPU util/mem: {_gpu}")
            # RAM の内訳も毎回残す。重みロードで固まる原因は「空きが少ない」
            # だけでは特定できない（cache なのか anon なのか /dev/shm なのか）。
            print(f"     {_mem_line()} / 上位: {_top_rss()}")
        time.sleep(5)

    # **なぜ待つのをやめたか**を残す。これが無いと、予算で切った回も
    # 「起動に失敗しました」と報告され、設定の誤りを疑って何時間も溶かす
    # （実測 2026-11-02: 予算切れを起動失敗と読んで原因究明に 1 時間）。
    if started:
        _why = "started"
    elif proc.poll() is not None:
        _why = "died"
    else:
        _why = "budget"
    _PF_STATE = {"requested": bool(LOAD_ARGS),
                 "args": LOAD_ARGS,
                 "declined": bool(_PF_GUARD_HIT),
                 "declined_line": _PF_GUARD_HIT}
    if _PF_STATE["requested"]:
        print(("⚠ prefetch は辞退された（この回の重みロードは遅い経路）"
               if _PF_STATE["declined"] else "ℹ prefetch は辞退されていない"))
        kc_save_timings({"prefetch": _PF_STATE})

    smoke_fail = ""
    if started:
        # 起動できても初回 forward でカーネルが落ちる領域（sm75 + 量子化 +
        # GDN Triton）があるため、1 回生成させて通ったときだけ成功とする。
        # smoke の待ち時間も総予算から差し引く（予算超過で次段が走らない事故を防ぐ）。
        _remain = int(STARTUP_BUDGET - (time.time() - _t_start))
        ok, detail = _smoke(timeout=max(30, min(90, _remain)))
        if ok:
            argv_used = cmd
            print(f"✅ vLLM 起動+生成 OK（{int(time.time()-t0)}s, 試行 {idx}）"
                  f" {detail}")
            # 起動の内訳を残す（高度化の基礎）。これが無いと毎回「なんとなく
            # 遅い」で終わり、施策の効果を比べられない。
            _load_s = round(time.time() - t0, 1)
            _span = None
            _gaps = []
            _ok_log = ""
            try:
                _ok_log = open(log_path, errors="replace").read()
                _marks = _phase_marks(_ok_log, _t0_wall)
                # ログ時刻（1 秒精度）を優先し、無い節目はポーリング（60 秒）で補う。
                # graph capture は tqdm 行なので時刻が無く、ポーリング側にしか出ない。
                _marks = {**_poll_marks, **_marks}
                _span = _phase_spans(_marks)
                _gaps = _long_gaps(_ok_log, _t0_wall)
                print(f"⏱ 起動の節目: {_marks}")
                print(f"⏱ 区間: {_span}")
                for _g in _gaps:
                    print(f"⏱ 無音 {_g['seconds']}s @{_g['from']}s: "
                          f"「{_g['before'][:90]}」→「{_g['after'][:90]}」")
            except Exception as _e:
                _marks = {"error": f"{type(_e).__name__}: {_e}"}
            # **成功時こそログを残す。** 旧実装は失敗時しか保存しておらず、
            # 「compile 後 379 秒の無音区間」の中身を読めなかった（2026-11-01）。
            # 起動できた回のログは 200 KiB 程度なので、毎回残して損はない。
            if _ok_log:
                save_output(f"vllm_{idx}.log", _ok_log[-3000000:])
            kc_save_timings({"load_seconds": _load_s,
                             "attempt": {"index": idx, "label": label,
                                         "max_model_len": maxlen},
                             "startup_marks": _marks,
                             "startup_spans": _span,
                             "startup_gaps": _gaps})
            # キャッシュが**当たったか**を判定する。「ファイルがある」ではなく
            # 「エントリが増えなかった」。373 秒の無音が戻ったときに、ミスなのか
            # 別の原因なのかをここで切り分けられるようにする。
            _tr = _triton_report(_TRITON_BEFORE, _triton_snapshot())
            print(f"🧠 Triton キャッシュ: {_tr['entries_before']} → "
                  f"{_tr['entries_after']} エントリ"
                  f"（新規 {_tr['new_entries']} / 再利用率 {_tr['reused_pct']}%）")
            if _tr["new_entries"] and not _tr["entries_before"]:
                print("   ℹ 前回のキャッシュが無いので今回は全部作り直し"
                      "（この tar を Dataset 化すれば次回効く）")
            elif (_tr["entries_before"]
                  and _tr["new_entries"] > _tr["entries_before"] * 0.2):
                print("   ⚠ 新規エントリが多い＝キャッシュが当たっていない"
                      "（Triton の版・sm・ソースが前回と違う）")
            kc_save_timings({"triton_cache": _tr})
            break
        print(f"⚠ 起動はしたが生成 NG → 次段へ: {detail[:300]}")
        smoke_fail = detail[:800]
        started = False

    tail = ""
    try:
        _full = open(log_path, errors="replace").read()
    except Exception:
        _full = ""
    tail = _full[-4000:]
    # 末尾だけでは「Engine core initialization failed. See root cause above.」
    # のように**原因が上にある**ケースを読めない（2026-10-29 実測）。
    # 失敗ログは丸ごと成果物に残し、原因らしき行も抜き出して表示する。
    if _full:
        save_output(f"vllm_{idx}.log", _full[-3000000:])
    _keys = ("Traceback", "Error", "error", "OutOfMemory", "OOM",
             "RuntimeError", "assert", "CUDA", "not supported",
             "No available shared memory")
    _sig_lines = [ln.strip() for ln in _full.splitlines()
                  if any(k in ln for k in _keys)]
    if _sig_lines:
        print("❌ 起動失敗。原因候補（ログ内の該当行, 先頭20）:")
        for _l in _sig_lines[:20]:
            print("   | " + _l[:200])
    print("❌ 起動失敗。ログ末尾:")
    print(tail)
    # 途中で GPU クォータが尽きても原因が残るよう、失敗ごとに追記保存する。
    _attempts.append(f"--- 試行 {idx} {label} max_len={maxlen} "
                     f"opt={'on' if use_opt else 'off'} ---\n"
                     + (f"[smoke NG] {smoke_fail}\n" if smoke_fail else "")
                     + tail)
    save_output("vllm_attempts.txt", "\n\n".join(_attempts) + "\n")
    _kill(proc)
    # 同一エラーで落ち続けるなら、フラグを変えても無駄。予算を残して止める。
    # 末尾 3 行を署名にする。最終行だけだと TypeError 等の一般論で一致して
    # 別原因の失敗まで打ち切ってしまう（残りクォータを捨てる損失が大きい）。
    _sig = _sig_of(tail)
    # smoke 失敗はサーバーログ末尾が正常行になるため、署名一致で打ち切ると
    # 別段の失敗まで巻き込む。smoke 失敗時は次段へ必ず進む。
    if _sig and _sig == _prev_sig and not smoke_fail:
        print(f"⛔ 同一エラーで再失敗 → ラダー打ち切り: {_sig}")
        break
    _prev_sig = _sig

# 起動中の RAM 推移を残す（成功・失敗どちらでも）。24 GiB が消える瞬間を
# 捕まえるための唯一の記録で、これが無いと毎回「謎のハング」に戻る。
_RAM_STOP.set()
_RAM_THREAD.join(timeout=10)
try:
    _rt = open(RAM_TRACE, encoding="utf-8").read()
    if _rt.strip():
        save_output("ram_trace.txt", _rt)
        print(f"  📄 ram_trace.txt 保存済（{len(_rt)} B）")
except OSError:
    pass

# ---- 次回用に Triton キャッシュを退避 ----
# 成功・失敗の**合流点**に置く。失敗した回でも、373 秒かけて作ったカーネルは
# ディスクに残っているので、tar にして持ち出せば次回は当たる。既存の
# `engine_cleanup()`（キープアライブ終了時）は v14 で一度も走らなかったため、
# 起動シーケンスの中で確実に通るここで取る。
_WARM_CACHE = _capture_warm_cache()
if _WARM_CACHE.get("path"):
    print(f"  📦 warm-cache.tar {_WARM_CACHE['bytes']/1e6:.0f} MB"
          f"（{_WARM_CACHE['seconds']}s, 元 {_WARM_CACHE['source_bytes']/1e6:.0f} MB）"
          f" → Dataset 化すると次回の JIT を省ける")
elif _WARM_CACHE.get("skipped"):
    print(f"  ⏭ キャッシュ退避なし: {_WARM_CACHE['skipped']}")
kc_save_timings({"warm_cache": _WARM_CACHE})
if not started:
    save_output("vllm_startup.log", tail)
    save_output("vllm_attempts.txt", "\n\n".join(_attempts) + "\n")
    if _why == "budget":
        # 予算切れは**設定の誤りではない**。ここを「起動失敗」と報告すると、
        # 次の回にフラグを疑って同じ 70 分を無駄にする。最後の節目を添えて、
        # 「どこまで進んで切られたか」だけを言う。
        fail(f"起動が予算 {STARTUP_BUDGET}s 内に終わらなかった（設定の誤りでは"
             f"ない。遅いノードで重みロードが間に合わなかった）。"
             f"最後に越えた節目: {_mark_last}。"
             f"prefetch={'辞退された' if _PF_GUARD_HIT else '効いていた'}"
             f"（vllm_startup.log / vllm_attempts.txt 参照）",
             last_log or "/tmp/vllm_1.log")
    fail("vLLM の起動に失敗しました（vllm_startup.log / vllm_*.log 参照）",
         last_log or "/tmp/vllm_1.log")

# ---- 起動結果を保存 ----
info = _models() or {}
ids = [m.get("id") for m in info.get("data", [])]
max_len = (info.get("data") or [{}])[0].get("max_model_len")
# 共有セル（接続情報）が読む値。設定セルの候補値ではなく実測値で上書きする。
CONTEXT_LENGTH = max_len or CONTEXT_LENGTH
with open("/tmp/vllm_cmd.json", "w") as f:
    json.dump({"argv": argv_used, "spec": bool(SPEC), "models": ids,
               "max_model_len": max_len}, f, indent=2)
print()
print(f"  models        : {ids}")
print(f"  max_model_len : {max_len}")
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.strip())
save_output("vllm_cmd.json", json.dumps(
    {"argv": argv_used, "models": ids, "max_model_len": max_len},
    indent=2) + "\n")


---
### 🔥 ウォームアップ（初回生成は CUDA graph 構築で遅い）


In [ ]:
# ダミー生成で疎通確認 + 速度計測
import json
import subprocess
import time
import urllib.request

payload = {
    "model": SERVED_NAME,
    "messages": [{"role": "user", "content": "Say OK."}],
    "max_tokens": 64,
    "temperature": 0.7,
    "repetition_penalty": 1.05,
    "stream": False,
}
req = urllib.request.Request(
    f"http://127.0.0.1:{VLLM_PORT}/v1/chat/completions",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"},
    method="POST",
)
print("⏳ ウォームアップ中...")
t0 = time.time()
try:
    with urllib.request.urlopen(req, timeout=900) as r:
        resp = json.loads(r.read())
    dt = time.time() - t0
except Exception as e:
    # 起動ログは試行番号で名前が変わる。読めなくても失敗自体は伝える。
    for _i in range(1, 7):
        _p = "/tmp/vllm_%d.log" % _i
        try:
            print(f"--- {_p} ---")
            print(open(_p, errors="replace").read()[-2500:])
        except OSError:
            continue
    fail(f"ウォームアップ失敗: {type(e).__name__}: {e}")

msg = (resp.get("choices") or [{}])[0].get("message") or {}
content = msg.get("content") or ""
reasoning = msg.get("reasoning_content") or ""
usage = resp.get("usage") or {}
ct = usage.get("completion_tokens") or 0
print(f"✅ 応答あり（{dt:.1f}s）")
print(f"   reply : {content[:120]!r}")
print(f"   think : {reasoning[:80]!r}")
print(f"   tokens: completion={ct} total={usage.get('total_tokens')}")
if ct:
    print(f"   速度  : {ct/dt:.1f} tokens/s（ウォームアップ込み）")
if not (content or reasoning or ct):
    fail("生成結果が空です（プロンプト/チャットテンプレートを確認）")
# 速度も残す。ログにしか無いと「前回より速い/遅い」を比べられない。
kc_save_timings({"speed_tps": round(ct / dt, 1) if ct and dt else None,
                 "warmup_seconds": round(dt, 1)})

# GPU 常駐（CPU フォールバックでも HTTP 200 は返るので必ず見る）
mem = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.strip()
print(f"   VRAM  : {mem}")
# nvidia-smi の値は "N/A" や空になり得る。int() で落とさず、数字だけ拾う。
used = []
for _ln in mem.splitlines():
    _a = _ln.split(",")[0].strip()
    if _a.isdigit():
        used.append(int(_a))
if not used:
    # ここが旧実装の穴だった。`if used and ...` は **used が空だと素通り**するので、
    # nvidia-smi が何も返さない（= GPU 不在 / CPU 推論）ケースを検出できなかった。
    # 「エラーが出ずに黙って CPU 推論」は実測 0.9 t/s で、warning だけが手がかり。
    fail(f"nvidia-smi から VRAM を取得できません（mem={mem!r}）。CPU 実行の可能性。")
if sum(used) < 4000:
    fail(f"VRAM 使用量が少なすぎます（{used} MiB）。CPU 実行の可能性。")
kc_save_timings({"gpu_vram_mib": used})


---
### 🏁 smoke: ここで終了

起動と会話（ウォームアップセルの疎通確認）まで確認済み。
トンネル / ビーコン公開 / keepalive は CPU 側で検証済みのため省略します。


In [ ]:
# smoke: プロセスを止めて終了（公開しない）
print()
print("=" * 56)
print("🏁 smoke: トンネル / ビーコン / keepalive は省略")
print("=" * 56)
left = kc_stop_processes([
    ("-f auth_proxy.py", "auth_proxy"),
    ("-f vllm serve", "vllm"),
    ("-f VLLM::EngineCore", "engine-core"),
])
# 残存を「表示だけ」で流すと、GPU を掴んだまま完走扱いになる。
# 実際に解放されたかまで見て、駄目ならカーネルを失敗させる。
_idle = kc_wait_gpu_idle(limit_mib=500, timeout=60)
print(f"  残存: {left or 'なし'}")
print(f"  GPU : {'解放' if _idle >= 0 else '未解放（60 秒待っても使用中）'}")
save_output("smoke.txt", "leftover=" + (",".join(left) if left else "none") + "\n"
                         + f"gpu_idle_after={_idle}\n")
if left:
    fail("smoke 後にプロセスが残存しました: " + ", ".join(left))
if _idle < 0:
    fail("smoke 後に GPU が解放されません（nvidia-smi を確認）")
print("  ✅ smoke 完了（プロセス停止 / GPU 解放を確認）")
